# Домашнє завдання 4: DML, DDL, JOIN і агрегати на Olist у PostgreSQL

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IYgit/goit-rdb-hw-04/blob/master/HW4.ipynb)

**Датасет.** [Olist Brazilian E-commerce Public Dataset](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) —
близько 100 тис. замовлень бразильського e-commerce-майданчика Olist за 2016–2018 роки.
Використано **6 пов'язаних таблиць**: клієнти, замовлення, позиції замовлень, товари, продавці, відгуки.

**Роль і сценарій.** Працюємо як Data Engineer / Data Analyst: треба підготувати багатотабличний
dataset до аналітики й ML — забезпечити цілісність даних, побудувати коректні зв'язки, сформувати
ознаки на рівні покупців і продавців, порахувати бізнес-метрики.

**Наскрізна тема цього ДЗ — грануляність.** Таблиці Olist мають різний «крок»: позиція замовлення,
замовлення, відгук, покупець. Майже кожна помилка в такій схемі — це не синтаксис, а JOIN, який
непомітно множить рядки. Тому в роботі є два окремі докази того, що метрики не роздуті
(розділи 2.2 і 4.2), і вони ж пояснюють, чому `seller_score` рахується через
`seller_id × order_id`, а `customer_segments` — через `customer_unique_id`.

**Що в цьому notebook:**

| Розділ | Зміст |
|---|---|
| Завдання 1 | `pgserver`, відтворюване отримання Olist, raw-рівень через `COPY`, typed-схема з PK/FK/CHECK |
| Завдання 2 | DML: `INSERT ... RETURNING`, `ON CONFLICT DO UPDATE`, `UPDATE ... FROM`, `DELETE ... RETURNING` |
| Завдання 3 | `olist_customer_dim` і `customer_segments` на рівні `customer_unique_id` |
| Завдання 4 | 6 JOIN-запитів: INNER, LEFT+COALESCE, FULL OUTER, SELF, anti-join (`IS NULL` і `NOT EXISTS`) |
| Завдання 5 | 4 агреговані запити: `GROUP BY`, `HAVING`, `STRING_AGG`, `ARRAY_AGG`, `JSONB_AGG` |
| Завдання 6 | Set operations, `ROW_NUMBER` / `RANK`, автоперевірки, Reflection |

---

# Завдання 1. Setup і typed-схема Olist

## 1.1 Середовище й підключення

In [3]:
# ============================================================
# PostgreSQL + Python environment
# ============================================================

%pip install -q pgserver psycopg2-binary "sqlalchemy>=2.0" pandas pyarrow jupysql

In [5]:
# ============================================================
# Imports and configuration
# ============================================================

import gzip
import os
import shutil
import sys
import tempfile
from pathlib import Path
from urllib.request import urlopen

import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

IN_COLAB = "google.colab" in sys.modules
BASE_DIR = Path("/content") if IN_COLAB else Path.cwd()
DATA_DIR = BASE_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_rows", 12)
pd.set_option("display.max_colwidth", 70)


def ensure_pg_timezone_files():
    """Windows-колесо pgserver не містить share/postgresql/timezone.

    Без цього каталогу сервер стартує з TimeZone=GMT, але SET TIME ZONE 'UTC'
    падає. У Colab/Linux каталог уже є, тому функція нічого не робить.
    """
    tz_dir = Path(pgserver.__file__).parent / "pginstall" / "share" / "postgresql" / "timezone"
    if tz_dir.exists():
        return False
    import tzdata

    shutil.copytree(Path(tzdata.__file__).parent / "zoneinfo", tz_dir)
    return True


# ============================================================
# Start PostgreSQL through pgserver
# ============================================================

patched_tz = ensure_pg_timezone_files()
CLUSTER_DIR = Path(tempfile.gettempdir()) / "hw4_pg"

pg = pgserver.get_server(str(CLUSTER_DIR), cleanup_mode="stop")

# SQLAlchemy 2.1 віддає "postgresql://" драйверу psycopg (v3); фіксуємо psycopg2 явно,
# бо саме його cursor.copy_expert() використовується для COPY нижче.
engine = create_engine(
    pg.get_uri().replace("postgresql://", "postgresql+psycopg2://", 1),
    connect_args={"options": "-c timezone=UTC"},
    future=True,
)

with engine.connect() as conn:
    print(conn.execute(text("SELECT version()")).scalar_one())
    print("Timezone:", conn.execute(text("SELECT current_setting('timezone')")).scalar_one())

print("Google Colab:", IN_COLAB)
print("Каталог даних:", DATA_DIR)
print("Кластер PostgreSQL:", CLUSTER_DIR)
print("Скопійовано timezone-файли для Windows:", patched_tz)

PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit
Timezone: UTC
Google Colab: True
Каталог даних: /content/data
Кластер PostgreSQL: /tmp/hw4_pg
Скопійовано timezone-файли для Windows: False


SQL виконуємо магією `%%sql` (пакет **JupySQL**), результат повертається як `pandas.DataFrame`.

In [6]:
%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.displaycon = False
%config SqlMagic.feedback = 1
%sql engine

%sql SELECT current_setting('TimeZone') AS session_timezone, NOW() AS server_now;

1 rows affected.

,session_timezone,server_now
0,UTC,2026-09-29 10:14:44.003354+00:00


## 1.2 Відтворюване отримання даних Olist

Порядок джерел — від «як у методичці» до «працює завжди»:

1. **KaggleHub** — `olistbr/brazilian-ecommerce`. Використовується, лише якщо креденшели Kaggle
   справді налаштовані (`KAGGLE_USERNAME`/`KAGGLE_KEY` або `~/.kaggle/kaggle.json`). Перевірка
   потрібна, бо без токена `kagglehub` намагається спитати креденшели інтерактивно, а це зависання
   при *Restart & Run All*.
2. **Локальна папка `data/olist/`** — 6 файлів `*.csv.gz`, які лежать у репозиторії (26 МБ).
3. **Публічне дзеркало** тих самих файлів на HuggingFace — якщо репозиторій клоновано без даних.

Усі три шляхи дають **однакові файли**: перевірено, що в дзеркалі збережено навіть оригінальну
опиcку в назвах колонок `product_name_lenght` / `product_description_lenght`.

In [7]:
OLIST_FILES = {
    "customers":   "olist_customers_dataset.csv",
    "orders":      "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "products":    "olist_products_dataset.csv",
    "sellers":     "olist_sellers_dataset.csv",
    "reviews":     "olist_order_reviews_dataset.csv",
}
MIRROR_URL = "https://huggingface.co/datasets/bulutttt/olist-raw-data/resolve/main/"
OLIST_DIR = DATA_DIR / "olist"


def _kaggle_credentials_ready():
    if os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"):
        return True
    return (Path.home() / ".kaggle" / "kaggle.json").exists()


def resolve_olist_files():
    """Повертає (dict назва -> шлях, опис джерела)."""
    # 1. Kaggle
    if _kaggle_credentials_ready():
        try:
            import kagglehub

            kaggle_dir = Path(kagglehub.dataset_download("olistbr/brazilian-ecommerce"))
            paths = {k: kaggle_dir / v for k, v in OLIST_FILES.items()}
            if all(p.exists() for p in paths.values()):
                return paths, "kagglehub: olistbr/brazilian-ecommerce"
        except Exception as exc:  # немає мережі / токен відкликано
            print("KaggleHub недоступний, переходимо до наступного джерела:", exc)

    # 2. Локальні файли репозиторію
    OLIST_DIR.mkdir(parents=True, exist_ok=True)
    paths = {}
    for key, name in OLIST_FILES.items():
        gz_path, csv_path = OLIST_DIR / (name + ".gz"), OLIST_DIR / name
        paths[key] = gz_path if gz_path.exists() else csv_path
    if all(p.exists() for p in paths.values()):
        return paths, f"локальна папка {OLIST_DIR}"

    # 3. Публічне дзеркало
    for key, name in OLIST_FILES.items():
        dst = OLIST_DIR / (name + ".gz")
        if not dst.exists():
            print("завантаження", name)
            with urlopen(MIRROR_URL + name, timeout=300) as src, gzip.open(dst, "wb") as out:
                shutil.copyfileobj(src, out)
        paths[key] = dst
    return paths, "публічне дзеркало HuggingFace (bulutttt/olist-raw-data)"


OLIST_PATHS, OLIST_SOURCE = resolve_olist_files()
print("Джерело даних:", OLIST_SOURCE)
for key, path in OLIST_PATHS.items():
    print(f"  {key:12s} {path.name:40s} {path.stat().st_size / 1e6:6.2f} МБ")

завантаження olist_customers_dataset.csv
завантаження olist_orders_dataset.csv
завантаження olist_order_items_dataset.csv
завантаження olist_products_dataset.csv
завантаження olist_sellers_dataset.csv
завантаження olist_order_reviews_dataset.csv
Джерело даних: публічне дзеркало HuggingFace (bulutttt/olist-raw-data)
  customers    olist_customers_dataset.csv.gz             4.79 МБ
  orders       olist_orders_dataset.csv.gz                6.93 МБ
  order_items  olist_order_items_dataset.csv.gz           6.62 МБ
  products     olist_products_dataset.csv.gz              1.05 МБ
  sellers      olist_sellers_dataset.csv.gz               0.08 МБ
  reviews      olist_order_reviews_dataset.csv.gz         6.65 МБ


## 1.3 Raw-рівень

Raw-таблиці — проміжний шар: вони дзеркалять вихідні CSV (включно з опискою `lenght`) і **не**
замінюють typed-схему з обмеженнями. Типи колонок тут — ті, які виводить `pandas`.

Завантаження йде через `COPY ... FROM STDIN`, а не через `DataFrame.to_sql(...)`: `to_sql` шле
рядки по одному через `executemany` і на 112 тис. позицій замовлень займає хвилини, тоді як `COPY`
— секунди. Схема таблиці при цьому та сама: її створює `to_sql` порожнім кадром `df.head(0)`.

In [8]:
RAW_TABLES = {
    "customers":   "olist_customers_raw",
    "orders":      "olist_orders_raw",
    "order_items": "olist_order_items_raw",
    "products":    "olist_products_raw",
    "sellers":     "olist_sellers_raw",
    "reviews":     "olist_order_reviews_raw",
}


def load_raw(df, table):
    """DDL через to_sql(head(0)), дані — через COPY ... FROM STDIN."""
    import io

    df.head(0).to_sql(table, engine, if_exists="replace", index=False)

    buffer = io.StringIO()
    df.to_csv(buffer, index=False, header=False)
    buffer.seek(0)

    raw_conn = engine.raw_connection()
    try:
        with raw_conn.cursor() as cur:
            cur.copy_expert(f'COPY "{table}" FROM STDIN WITH (FORMAT csv)', buffer)
        raw_conn.commit()
    finally:
        raw_conn.close()


for key, table in RAW_TABLES.items():
    frame = pd.read_csv(OLIST_PATHS[key])
    load_raw(frame, table)
    print(f"{table:26s} {frame.shape[0]:>7,} рядків × {frame.shape[1]} колонок")

olist_customers_raw         99,441 рядків × 5 колонок
olist_orders_raw            99,441 рядків × 8 колонок
olist_order_items_raw      112,650 рядків × 7 колонок
olist_products_raw          32,951 рядків × 9 колонок
olist_sellers_raw            3,095 рядків × 4 колонок
olist_order_reviews_raw     99,224 рядків × 7 колонок


Перші рядки raw-рівня — саме те, що лежить у CSV:

In [9]:
%%sql
SELECT order_id, order_item_id, product_id, seller_id, price, freight_value
FROM olist_order_items_raw
ORDER BY order_id
LIMIT 5;

5 rows affected.

,order_id,order_item_id,product_id,seller_id,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,199.90,18.14


## 1.4 Typed-схема: ключі й обмеження цілісності

Рішення, які закладені в DDL нижче:

| Таблиця | Ключ | Головні обмеження | Чому саме так |
|---|---|---|---|
| `olist_customers` | `customer_id` | `customer_unique_id NOT NULL`, індекс по ньому | `customer_id` унікальний у джерелі; індекс потрібен, бо всі customer-level запити йдуть через `customer_unique_id` |
| `olist_orders` | `order_id` | FK на клієнта, `CHECK` на 8 дозволених статусів | статус — закритий словник, `CHECK` ловить друк чи новий код зі зміненого джерела |
| `olist_products` | `product_id` | 7 `CHECK (… IS NULL OR … >= 0)` | розміри й ваги не можуть бути відʼємні, але **можуть** бути `NULL` — 610 товарів без картки |
| `olist_sellers` | `seller_id` | `seller_state NOT NULL` | без штату продавець непридатний для регіональної аналітики |
| `olist_order_items` | `(order_id, order_item_id)` | 3 FK, `CHECK` на невідʼємні гроші | складений PK фіксує грануляність «позиція замовлення» |
| `olist_order_reviews` | **технічний** `review_row_id` | FK на замовлення, `CHECK (review_score BETWEEN 1 AND 5)` | `review_id` у джерелі **не унікальний** — доказ у 1.6 |

`NUMERIC(12,2)` для грошей, а не `FLOAT`: сума 112 650 позицій має бути точною до копійки.
`TIMESTAMP` без таймзони — Olist віддає наївний бразильський час, вигадувати йому зсув не варто.

In [10]:
%%sql
DROP TABLE IF EXISTS customer_segments    CASCADE;
DROP TABLE IF EXISTS seller_score         CASCADE;
DROP TABLE IF EXISTS seller_alerts        CASCADE;
DROP TABLE IF EXISTS olist_customer_dim   CASCADE;
DROP TABLE IF EXISTS olist_order_reviews  CASCADE;
DROP TABLE IF EXISTS olist_order_items    CASCADE;
DROP TABLE IF EXISTS olist_orders         CASCADE;
DROP TABLE IF EXISTS olist_customers      CASCADE;
DROP TABLE IF EXISTS olist_products       CASCADE;
DROP TABLE IF EXISTS olist_sellers        CASCADE;

SELECT COUNT(*) AS typed_tables_left
FROM information_schema.tables
WHERE table_schema = 'public'
  AND table_name IN (
      'olist_customers', 'olist_orders', 'olist_order_items',
      'olist_products', 'olist_sellers', 'olist_order_reviews',
      'olist_customer_dim', 'customer_segments', 'seller_score', 'seller_alerts'
  );

1 rows affected.

,typed_tables_left
0,0


Порожній результат `typed_tables_left = 0` означає, що клітинка нижче будує схему з нуля —
саме це робить *Restart & Run All* повторюваним.

In [11]:
%%sql
CREATE TABLE olist_customers (
    customer_id              TEXT PRIMARY KEY,
    customer_unique_id       TEXT NOT NULL,
    customer_zip_code_prefix INTEGER,
    customer_city            TEXT,
    customer_state           CHAR(2) NOT NULL
);

CREATE INDEX idx_olist_customers_unique_id
    ON olist_customers (customer_unique_id);

CREATE TABLE olist_orders (
    order_id                      TEXT PRIMARY KEY,
    customer_id                   TEXT NOT NULL REFERENCES olist_customers(customer_id),
    order_status                  TEXT NOT NULL,
    order_purchase_timestamp      TIMESTAMP NOT NULL,
    order_approved_at             TIMESTAMP,
    order_delivered_carrier_date  TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP,
    CHECK (order_status IN (
        'created', 'approved', 'invoiced', 'processing',
        'shipped', 'delivered', 'unavailable', 'canceled'
    ))
);

CREATE TABLE olist_products (
    product_id                 TEXT PRIMARY KEY,
    product_category_name      TEXT,
    product_name_length        INTEGER CHECK (product_name_length IS NULL OR product_name_length >= 0),
    product_description_length INTEGER CHECK (product_description_length IS NULL OR product_description_length >= 0),
    product_photos_qty         INTEGER CHECK (product_photos_qty IS NULL OR product_photos_qty >= 0),
    product_weight_g           INTEGER CHECK (product_weight_g IS NULL OR product_weight_g >= 0),
    product_length_cm          INTEGER CHECK (product_length_cm IS NULL OR product_length_cm >= 0),
    product_height_cm          INTEGER CHECK (product_height_cm IS NULL OR product_height_cm >= 0),
    product_width_cm           INTEGER CHECK (product_width_cm IS NULL OR product_width_cm >= 0)
);

CREATE TABLE olist_sellers (
    seller_id              TEXT PRIMARY KEY,
    seller_zip_code_prefix INTEGER,
    seller_city            TEXT,
    seller_state           CHAR(2) NOT NULL
);

CREATE TABLE olist_order_items (
    order_id            TEXT NOT NULL REFERENCES olist_orders(order_id),
    order_item_id       INTEGER NOT NULL,
    product_id          TEXT REFERENCES olist_products(product_id),
    seller_id           TEXT REFERENCES olist_sellers(seller_id),
    shipping_limit_date TIMESTAMP,
    price               NUMERIC(12, 2) CHECK (price IS NULL OR price >= 0),
    freight_value       NUMERIC(12, 2) CHECK (freight_value IS NULL OR freight_value >= 0),
    PRIMARY KEY (order_id, order_item_id)
);

CREATE INDEX idx_olist_order_items_seller_order
    ON olist_order_items (seller_id, order_id);

CREATE TABLE olist_order_reviews (
    review_row_id            BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    review_id                TEXT,
    order_id                 TEXT NOT NULL REFERENCES olist_orders(order_id),
    review_score             SMALLINT CHECK (review_score BETWEEN 1 AND 5),
    review_comment_title     TEXT,
    review_comment_message   TEXT,
    review_creation_date     TIMESTAMP,
    review_answer_timestamp  TIMESTAMP
);

CREATE INDEX idx_olist_order_reviews_order_id
    ON olist_order_reviews (order_id);

SELECT
    table_name,
    COUNT(*) AS n_columns
FROM information_schema.columns
WHERE table_schema = 'public'
  AND table_name IN (
      'olist_customers', 'olist_orders', 'olist_order_items',
      'olist_products', 'olist_sellers', 'olist_order_reviews'
  )
GROUP BY table_name
ORDER BY table_name;

6 rows affected.

,table_name,n_columns
0,olist_customers,5
1,olist_order_items,7
2,olist_order_reviews,8
3,olist_orders,8
4,olist_products,9
5,olist_sellers,4


## 1.5 Перенесення raw → typed

`INSERT ... SELECT` із явним переліком колонок: типи приводяться тут, а не в pandas, тому будь-яке
значення, що не проходить `CHECK` чи FK, зупинило б завантаження.

Зверніть увагу на `product_name_lenght` і `product_description_lenght` у `SELECT`: у вихідному CSV
ці назви написані з опискою, а в typed-схемі вони нормалізовані до `…_length`.

Кількість рядків кожного `INSERT` видно у виводі, а закриваючий `SELECT` показує період, який
покриває завантажений датасет.

In [12]:
%%sql
INSERT INTO olist_customers (
    customer_id, customer_unique_id, customer_zip_code_prefix,
    customer_city, customer_state
)
SELECT
    customer_id,
    customer_unique_id,
    customer_zip_code_prefix::INTEGER,
    customer_city,
    customer_state::CHAR(2)
FROM olist_customers_raw;

INSERT INTO olist_orders (
    order_id, customer_id, order_status, order_purchase_timestamp,
    order_approved_at, order_delivered_carrier_date,
    order_delivered_customer_date, order_estimated_delivery_date
)
SELECT
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp::TIMESTAMP,
    order_approved_at::TIMESTAMP,
    order_delivered_carrier_date::TIMESTAMP,
    order_delivered_customer_date::TIMESTAMP,
    order_estimated_delivery_date::TIMESTAMP
FROM olist_orders_raw;

INSERT INTO olist_products (
    product_id, product_category_name, product_name_length,
    product_description_length, product_photos_qty, product_weight_g,
    product_length_cm, product_height_cm, product_width_cm
)
SELECT
    product_id,
    product_category_name,
    product_name_lenght::INTEGER,
    product_description_lenght::INTEGER,
    product_photos_qty::INTEGER,
    product_weight_g::INTEGER,
    product_length_cm::INTEGER,
    product_height_cm::INTEGER,
    product_width_cm::INTEGER
FROM olist_products_raw;

INSERT INTO olist_sellers (
    seller_id, seller_zip_code_prefix, seller_city, seller_state
)
SELECT
    seller_id,
    seller_zip_code_prefix::INTEGER,
    seller_city,
    seller_state::CHAR(2)
FROM olist_sellers_raw;

INSERT INTO olist_order_items (
    order_id, order_item_id, product_id, seller_id,
    shipping_limit_date, price, freight_value
)
SELECT
    order_id,
    order_item_id::INTEGER,
    product_id,
    seller_id,
    shipping_limit_date::TIMESTAMP,
    price::NUMERIC(12, 2),
    freight_value::NUMERIC(12, 2)
FROM olist_order_items_raw;

INSERT INTO olist_order_reviews (
    review_id,
    order_id,
    review_score,
    review_comment_title,
    review_comment_message,
    review_creation_date,
    review_answer_timestamp
)
SELECT
    review_id,
    order_id,
    review_score::SMALLINT,
    review_comment_title,
    review_comment_message,
    review_creation_date::TIMESTAMP,
    review_answer_timestamp::TIMESTAMP
FROM olist_order_reviews_raw;

SELECT
    MIN(order_purchase_timestamp)::DATE AS first_order,
    MAX(order_purchase_timestamp)::DATE AS last_order,
    COUNT(DISTINCT order_status)        AS distinct_statuses
FROM olist_orders;

99441 rows affected.

99441 rows affected.

32951 rows affected.

3095 rows affected.

112650 rows affected.

99224 rows affected.

1 rows affected.

,first_order,last_order,distinct_statuses
0,2016-09-04,2018-10-17,8


## 1.6 Smoke-тести

Перше — чи не загубився жоден рядок при переході raw → typed.

In [13]:
%%sql
WITH raw_counts AS (
    SELECT 'customers'   AS table_name, COUNT(*) AS raw_rows FROM olist_customers_raw
    UNION ALL SELECT 'order_items', COUNT(*) FROM olist_order_items_raw
    UNION ALL SELECT 'orders',      COUNT(*) FROM olist_orders_raw
    UNION ALL SELECT 'products',    COUNT(*) FROM olist_products_raw
    UNION ALL SELECT 'reviews',     COUNT(*) FROM olist_order_reviews_raw
    UNION ALL SELECT 'sellers',     COUNT(*) FROM olist_sellers_raw
),
typed_counts AS (
    SELECT 'customers'   AS table_name, COUNT(*) AS typed_rows FROM olist_customers
    UNION ALL SELECT 'order_items', COUNT(*) FROM olist_order_items
    UNION ALL SELECT 'orders',      COUNT(*) FROM olist_orders
    UNION ALL SELECT 'products',    COUNT(*) FROM olist_products
    UNION ALL SELECT 'reviews',     COUNT(*) FROM olist_order_reviews
    UNION ALL SELECT 'sellers',     COUNT(*) FROM olist_sellers
)
SELECT
    r.table_name,
    r.raw_rows,
    t.typed_rows,
    t.typed_rows - r.raw_rows AS diff
FROM raw_counts AS r
JOIN typed_counts AS t USING (table_name)
ORDER BY r.table_name;

6 rows affected.

,table_name,raw_rows,typed_rows,diff
0,customers,99441,99441,0
1,order_items,112650,112650,0
2,orders,99441,99441,0
3,products,32951,32951,0
4,reviews,99224,99224,0
5,sellers,3095,3095,0


**Висновок 1.6a.** `diff = 0` для всіх шести таблиць: 99 441 клієнт, 99 441 замовлення,
112 650 позицій, 32 951 товар, 3 095 продавців, 99 224 відгуки — стільки ж, скільки в CSV.
Жоден рядок не відкинуто, і це не випадковість: усі п'ять звʼязків датасету цілісні, тому FK
не мали чого блокувати. Якби typed-таблиця виявилась меншою, причину треба було б шукати саме
в порушенні FK або `CHECK`.

Друге — чому `review_id` **не** можна робити первинним ключем.

In [14]:
%%sql
WITH per_order AS (
    SELECT order_id, COUNT(*) AS reviews_per_order
    FROM olist_order_reviews
    GROUP BY order_id
)
SELECT
    (SELECT COUNT(*) FROM olist_order_reviews)                    AS review_rows,
    (SELECT COUNT(DISTINCT review_id) FROM olist_order_reviews)   AS distinct_review_id,
    (SELECT COUNT(*) - COUNT(DISTINCT review_id)
       FROM olist_order_reviews)                                  AS duplicate_review_id_rows,
    (SELECT COUNT(*) FROM per_order)                              AS distinct_order_id,
    (SELECT COUNT(*) FROM per_order WHERE reviews_per_order > 1)  AS orders_with_many_reviews,
    (SELECT MAX(reviews_per_order) FROM per_order)                AS max_reviews_one_order;

1 rows affected.

,review_rows,distinct_review_id,duplicate_review_id_rows,distinct_order_id,orders_with_many_reviews,max_reviews_one_order
0,99224,98410,814,98673,547,3


**Висновок 1.6b.** У 99 224 рядках відгуків лише **98 410 унікальних `review_id`**, тобто
**814 рядків** — повтори ідентифікатора. Різних `order_id` ще менше — 98 673, і **547 замовлень**
мають більше одного відгуку (максимум — **3** на одне замовлення).

Отже `PRIMARY KEY (review_id)` впав би на цій копії датасету, а `PRIMARY KEY (order_id)` —
тим більше. Технічний `review_row_id BIGINT GENERATED BY DEFAULT AS IDENTITY` приймає дані
як є, зберігає `review_id` як звичайний джерельний атрибут і не змушує нас мовчки видаляти
рядки заради ключа. Ціна рішення — далі не можна вважати, що «один відгук = одне замовлення»:
перед будь-якою агрегацією відгуки треба зводити до рівня `order_id` (розділи 2.2 і 4.2).

Третє — інвентар обмежень, який підтверджує вимоги до DDL.

In [15]:
%%sql
SELECT
    c.relname AS table_name,
    COUNT(*) FILTER (WHERE con.contype = 'p') AS pk,
    COUNT(*) FILTER (WHERE con.contype = 'f') AS fk,
    COUNT(*) FILTER (WHERE con.contype = 'u') AS uniq,
    COUNT(*) FILTER (WHERE con.contype = 'c') AS checks,
    (SELECT COUNT(*) FROM information_schema.columns col
      WHERE col.table_schema = 'public'
        AND col.table_name = c.relname
        AND col.is_nullable = 'NO') AS not_null_columns
FROM pg_class AS c
JOIN pg_namespace AS n ON n.oid = c.relnamespace
LEFT JOIN pg_constraint AS con ON con.conrelid = c.oid
WHERE n.nspname = 'public'
  AND c.relkind = 'r'
  AND c.relname IN (
      'olist_customers', 'olist_orders', 'olist_order_items',
      'olist_products', 'olist_sellers', 'olist_order_reviews'
  )
GROUP BY c.relname, c.oid
ORDER BY c.relname;

6 rows affected.

,table_name,pk,fk,uniq,checks,not_null_columns
0,olist_customers,1,0,0,0,3
1,olist_order_items,1,3,0,2,2
2,olist_order_reviews,1,1,0,1,2
3,olist_orders,1,1,0,1,4
4,olist_products,1,0,0,7,1
5,olist_sellers,1,0,0,0,2


**Висновок 1.6c.** Схема має **6 `PRIMARY KEY`**, **5 `FOREIGN KEY`**, **11 `CHECK`** і
14 `NOT NULL`-колонок. Найбільше перевірок у `olist_products` (7) — там усі числові поля
описані як «або `NULL`, або невідʼємне»; у `olist_order_items` три FK (на замовлення, товар
і продавця) роблять цю таблицю справжнім центром схеми.

---

# Завдання 2. DML-запити з PostgreSQL-фічами

## 2.1 Пастка грануляності — перед тим, як рахувати `seller_score`

Перш ніж писати upsert, треба показати, чому «наївний» варіант із методички неправильний.
Відгук в Olist привʼязаний до **замовлення**, а `olist_order_items` має грануляність
**позиція замовлення**. Тому join `items ⋈ reviews` повторює один і той самий відгук стільки
разів, скільки позицій було в замовленні.

Запит нижче рахує обидва варіанти поруч.

In [16]:
%%sql
WITH naive AS (
    SELECT
        oi.seller_id,
        COUNT(rv.review_score)                  AS n_review_rows,
        AVG(rv.review_score)::NUMERIC(3, 2)     AS avg_rating
    FROM olist_order_items AS oi
    LEFT JOIN olist_order_reviews AS rv
        ON rv.order_id = oi.order_id
    GROUP BY oi.seller_id
),
seller_orders AS (
    SELECT DISTINCT oi.seller_id, oi.order_id
    FROM olist_order_items AS oi
),
review_per_order AS (
    SELECT rv.order_id, AVG(rv.review_score)::NUMERIC(3, 2) AS order_review_score
    FROM olist_order_reviews AS rv
    WHERE rv.review_score IS NOT NULL
    GROUP BY rv.order_id
),
correct AS (
    SELECT
        so.seller_id,
        COUNT(rpo.order_id)                          AS n_reviews,
        AVG(rpo.order_review_score)::NUMERIC(3, 2)   AS avg_rating
    FROM seller_orders AS so
    LEFT JOIN review_per_order AS rpo
        ON rpo.order_id = so.order_id
    GROUP BY so.seller_id
)
SELECT
    (SELECT SUM(n_review_rows) FROM naive)      AS naive_review_rows,
    (SELECT SUM(n_reviews) FROM correct)        AS correct_review_rows,
    (SELECT SUM(n_review_rows) FROM naive)
        - (SELECT SUM(n_reviews) FROM correct)  AS inflated_by,
    ROUND(
        100.0 * (SELECT SUM(n_review_rows) FROM naive)
              / NULLIF((SELECT SUM(n_reviews) FROM correct), 0) - 100, 2
    )                                           AS inflation_pct,
    (SELECT COUNT(*) FROM naive AS n JOIN correct AS c USING (seller_id)
      WHERE n.avg_rating IS DISTINCT FROM c.avg_rating) AS sellers_with_different_avg;

1 rows affected.

,naive_review_rows,correct_review_rows,inflated_by,inflation_pct,sellers_with_different_avg
0,112372,99247,13125,13.22,1264


**Висновок 2.1.** Наївний item-level join дає **112 372** review-рядки замість **99 247** —
це **+13 125 рядків, або +13.22 %** зайвого «ваги». І це не рівномірна похибка: у **1 264**
продавців із 3 095 (**41 %**) середній рейтинг просто інший, бо відгуки із замовлень на кілька
позицій ураховані по кілька разів. Продавець, у якого багато великих кошиків, отримав би
викривлений рейтинг.

Правильний порядок дій: (1) звести позиції до **`seller_id × order_id`** через `DISTINCT`,
(2) окремо звести відгуки до рівня **`order_id`**, (3) тільки потім рахувати seller-level метрики.
Саме так побудовано `seller_score` у 2.3.

## 2.2 `INSERT ... RETURNING` — alert для продавця з найбільшою кількістю запізнень

In [17]:
%%sql
DROP TABLE IF EXISTS seller_alerts CASCADE;

CREATE TABLE seller_alerts (
    alert_id   BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    seller_id  TEXT NOT NULL REFERENCES olist_sellers(seller_id),
    alert_type TEXT NOT NULL CHECK (alert_type IN ('late_delivery', 'low_rating', 'data_quality')),
    severity   SMALLINT NOT NULL CHECK (severity BETWEEN 1 AND 5),
    details    TEXT,
    created_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
);

SELECT conname, contype, pg_get_constraintdef(oid) AS definition
FROM pg_constraint
WHERE conrelid = 'seller_alerts'::regclass
ORDER BY contype, conname;

4 rows affected.

,conname,contype,definition
0,seller_alerts_alert_type_check,c,"CHECK ((alert_type = ANY (ARRAY['late_delivery'::text, 'low_rating..."
1,seller_alerts_severity_check,c,CHECK (((severity >= 1) AND (severity <= 5)))
2,seller_alerts_seller_id_fkey,f,FOREIGN KEY (seller_id) REFERENCES olist_sellers(seller_id)
3,seller_alerts_pkey,p,PRIMARY KEY (alert_id)


In [18]:
%%sql
WITH seller_orders AS (
    SELECT DISTINCT
        oi.seller_id,
        oi.order_id
    FROM olist_order_items AS oi
),
late_sellers AS (
    SELECT
        so.seller_id,
        COUNT(DISTINCT o.order_id) AS late_orders
    FROM seller_orders AS so
    JOIN olist_orders AS o
        ON o.order_id = so.order_id
    WHERE o.order_delivered_customer_date IS NOT NULL
      AND o.order_estimated_delivery_date IS NOT NULL
      AND o.order_delivered_customer_date::DATE > o.order_estimated_delivery_date::DATE
    GROUP BY so.seller_id
    ORDER BY late_orders DESC, so.seller_id
    LIMIT 1
)
INSERT INTO seller_alerts (seller_id, alert_type, severity, details)
SELECT
    seller_id,
    'late_delivery',
    4,
    'Seller has ' || late_orders || ' late delivered orders in the loaded dataset'
FROM late_sellers
RETURNING alert_id, seller_id, alert_type, severity, details, created_at;

1 rows affected.

,alert_id,seller_id,alert_type,severity,details,created_at
0,1,4a3ca9315b744ce9f8e9374361493884,late_delivery,4,Seller has 172 late delivered orders in the loaded dataset,2026-09-29 10:25:19.262156+00:00


**Що робить запит.** CTE `seller_orders` зводить позиції до пар «продавець × замовлення»
(`DISTINCT` — щоб замовлення з 5 позицій не рахувалося як 5 запізнень), `late_sellers` бере
одного найгіршого за кількістю замовлень, доставлених пізніше обіцяної дати, і `INSERT ... SELECT`
створює для нього alert.

**Результат.** Продавець `4a3ca931…` має **172 запізнілі доставки** — це найбільше в датасеті.
Далі в 6.3 виявиться, що це водночас **3-й продавець за GMV**: тобто проблема не в дрібному
продавцеві, а в одному з найбільших, і його SLA — тема для операційного рішення, а не для фільтра.

**Навіщо тут `RETURNING`.** `alert_id` генерується базою (`GENERATED ALWAYS AS IDENTITY`), а
`created_at` — `DEFAULT NOW()`. Без `RETURNING` довелось би робити другий запит, щоб дізнатися,
що саме записалось, і між ними міг би втрутитися інший процес. `RETURNING` віддає підтверджений
серверний стан рядка в тій самій транзакції — це готовий audit-запис: «о такій-то годині створено
alert №1 такого типу для такого продавця». У notebook він одразу видно як DataFrame.

## 2.3 `INSERT ... ON CONFLICT DO UPDATE` — idempotent upsert `seller_score`

In [19]:
%%sql
DROP TABLE IF EXISTS seller_score CASCADE;

CREATE TABLE seller_score (
    seller_id   TEXT PRIMARY KEY REFERENCES olist_sellers(seller_id),
    avg_rating  NUMERIC(3, 2),
    n_reviews   INTEGER NOT NULL DEFAULT 0 CHECK (n_reviews >= 0),
    tier        TEXT CHECK (tier IN ('gold', 'silver', 'bronze')),
    updated_at  TIMESTAMPTZ NOT NULL DEFAULT NOW()
);

SELECT conname, contype, pg_get_constraintdef(oid) AS definition
FROM pg_constraint
WHERE conrelid = 'seller_score'::regclass
ORDER BY contype, conname;

4 rows affected.

,conname,contype,definition
0,seller_score_n_reviews_check,c,CHECK ((n_reviews >= 0))
1,seller_score_tier_check,c,"CHECK ((tier = ANY (ARRAY['gold'::text, 'silver'::text, 'bronze'::..."
2,seller_score_seller_id_fkey,f,FOREIGN KEY (seller_id) REFERENCES olist_sellers(seller_id)
3,seller_score_pkey,p,PRIMARY KEY (seller_id)


In [20]:
%%sql
WITH seller_orders AS (
    SELECT DISTINCT
        oi.seller_id,
        oi.order_id
    FROM olist_order_items AS oi
),
review_per_order AS (
    SELECT
        rv.order_id,
        AVG(rv.review_score)::NUMERIC(3, 2) AS order_review_score
    FROM olist_order_reviews AS rv
    WHERE rv.review_score IS NOT NULL
    GROUP BY rv.order_id
),
seller_rating AS (
    SELECT
        so.seller_id,
        AVG(rpo.order_review_score)::NUMERIC(3, 2) AS avg_rating,
        COUNT(rpo.order_id) AS n_reviews
    FROM seller_orders AS so
    LEFT JOIN review_per_order AS rpo
        ON rpo.order_id = so.order_id
    GROUP BY so.seller_id
)
INSERT INTO seller_score (seller_id, avg_rating, n_reviews, tier)
SELECT
    seller_id,
    avg_rating,
    n_reviews,
    CASE
        WHEN avg_rating IS NULL THEN NULL
        WHEN avg_rating >= 4.5 THEN 'gold'
        WHEN avg_rating >= 3.5 THEN 'silver'
        ELSE 'bronze'
    END AS tier
FROM seller_rating
ON CONFLICT (seller_id) DO UPDATE
SET avg_rating = EXCLUDED.avg_rating,
    n_reviews  = EXCLUDED.n_reviews,
    tier       = EXCLUDED.tier,
    updated_at = NOW()
RETURNING seller_id, avg_rating, n_reviews, tier, updated_at;

3095 rows affected.

,seller_id,avg_rating,n_reviews,tier,updated_at
0,a61cc04793308395a840807104365121,5.00,1,gold,2026-09-29 10:26:14.242062+00:00
1,6d2f2e3b539480db1e0842b3a4e32e6e,3.67,3,silver,2026-09-29 10:26:14.242062+00:00
2,c53bcd3be457a342a97e39e5a9f0be22,3.20,5,bronze,2026-09-29 10:26:14.242062+00:00
3,9591fc341b1bfb7ef561e2968ec6e011,1.00,1,bronze,2026-09-29 10:26:14.242062+00:00
4,f9ec7093df3a7b346b7bcf7864069ca3,4.78,9,gold,2026-09-29 10:26:14.242062+00:00
...,...,...,...,...,...
3090,b2eecf5ea250510da76590ca79d60e5d,5.00,9,gold,2026-09-29 10:26:14.242062+00:00
3091,eae9af4811c294c56795d70e715b7337,5.00,1,gold,2026-09-29 10:26:14.242062+00:00
3092,1c83418358f249c9d6affa52d579db0e,5.00,1,gold,2026-09-29 10:26:14.242062+00:00
3093,845e8036c3f7bf62099352f5ed910240,3.95,21,silver,2026-09-29 10:26:14.242062+00:00


Розподіл по tier і момент останнього оновлення:

In [21]:
%%sql
SELECT
    COALESCE(tier, '(no reviews)')                       AS tier,
    COUNT(*)                                             AS n_sellers,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2)   AS pct_sellers,
    SUM(n_reviews)                                       AS total_reviews,
    ROUND(AVG(avg_rating), 3)                            AS mean_of_seller_avg,
    COUNT(DISTINCT updated_at)                           AS distinct_updated_at,
    MAX(updated_at)                                      AS updated_at
FROM seller_score
GROUP BY COALESCE(tier, '(no reviews)')
ORDER BY n_sellers DESC;

4 rows affected.

,tier,n_sellers,pct_sellers,total_reviews,mean_of_seller_avg,distinct_updated_at,updated_at
0,silver,1568,50.66,87123,4.063,1,2026-09-29 10:26:14.242062+00:00
1,gold,983,31.76,7093,4.827,1,2026-09-29 10:26:14.242062+00:00
2,bronze,539,17.42,5031,2.301,1,2026-09-29 10:26:14.242062+00:00
3,(no reviews),5,0.16,0,None,1,2026-09-29 10:26:14.242062+00:00


**Висновок 2.3.** Пораховано **3 095 продавців**: **silver 1 568 (50.7 %)**, **gold 983 (31.8 %)**,
**bronze 539 (17.4 %)** і **5 продавців зовсім без відгуків**. Цікаве тут — розподіл ваги:
на gold припадає лише **7 093** відгуки, а на silver — **87 123**. Тобто «золото» здебільшого
отримують дрібні продавці з 1–9 відгуками, де один задоволений покупець дає середнє 5.00, а великі
продавці майже завжди опиняються в silver. Використовувати `tier` як ML-ознаку без урахування
`n_reviews` не можна — це радше індикатор малої вибірки, ніж якості.

Відповіді на питання методички:

* **Чому `seller_orders` використовує `SELECT DISTINCT`** — щоб згорнути позиції до пар
  «продавець × замовлення». Без цього замовлення з кількома позиціями того самого продавця
  враховувалось би кілька разів. Це перший із двох кроків, що знімають множення (доказ у 2.1).
* **Чому review спершу агрегується до `order_id`** — бо 547 замовлень мають 2–3 відгуки
  (розділ 1.6b). `review_per_order` зводить їх до одного числа на замовлення, тому кожне
  замовлення заходить у seller-метрику з вагою 1.
* **Чому `ON CONFLICT (seller_id) DO UPDATE` робить запуск повторюваним** — ключ конфлікту збігається
  з `PRIMARY KEY`, тож повторний прогін не створює других рядків, а перезаписує наявні
  й оновлює `updated_at`. Це стандартний патерн idempotent-ETL: скільки б разів не запустили,
  таблиця має рівно один актуальний рядок на продавця.
* **Чому бувають `avg_rating = NULL`, `n_reviews = 0`, `tier = NULL`** — `seller_orders LEFT JOIN
  review_per_order` зберігає продавців, у чиїх замовленнях відгуків не було. `COUNT(rpo.order_id)`
  не рахує `NULL`, тому дає 0, `AVG` від порожнього набору — `NULL`, і `CASE` свідомо лишає `tier`
  порожнім. Це чесніше, ніж підставити 0 і зробити з «немає даних» найгіршу оцінку.

**Обмеження датасету.** Review в Olist привʼязаний до замовлення, а не до конкретного продавця.
Якщо в замовленні товари кількох продавців, один і той самий order-level відгук зарахується
кожному з них. Це прийнятне навчальне спрощення, але для production-скорингу потрібен був би
або розподіл ваги між продавцями, або окреме джерело seller-level відгуків.

### Демонстрація idempotency: той самий upsert ще раз

Клітинка нижче — **дослівно той самий запит**. Якщо `ON CONFLICT` працює, кількість рядків
не зміниться, а `updated_at` зсунеться.

In [22]:
%%sql
WITH seller_orders AS (
    SELECT DISTINCT
        oi.seller_id,
        oi.order_id
    FROM olist_order_items AS oi
),
review_per_order AS (
    SELECT
        rv.order_id,
        AVG(rv.review_score)::NUMERIC(3, 2) AS order_review_score
    FROM olist_order_reviews AS rv
    WHERE rv.review_score IS NOT NULL
    GROUP BY rv.order_id
),
seller_rating AS (
    SELECT
        so.seller_id,
        AVG(rpo.order_review_score)::NUMERIC(3, 2) AS avg_rating,
        COUNT(rpo.order_id) AS n_reviews
    FROM seller_orders AS so
    LEFT JOIN review_per_order AS rpo
        ON rpo.order_id = so.order_id
    GROUP BY so.seller_id
)
INSERT INTO seller_score (seller_id, avg_rating, n_reviews, tier)
SELECT
    seller_id,
    avg_rating,
    n_reviews,
    CASE
        WHEN avg_rating IS NULL THEN NULL
        WHEN avg_rating >= 4.5 THEN 'gold'
        WHEN avg_rating >= 3.5 THEN 'silver'
        ELSE 'bronze'
    END AS tier
FROM seller_rating
ON CONFLICT (seller_id) DO UPDATE
SET avg_rating = EXCLUDED.avg_rating,
    n_reviews  = EXCLUDED.n_reviews,
    tier       = EXCLUDED.tier,
    updated_at = NOW()
RETURNING seller_id, avg_rating, n_reviews, tier, updated_at;

3095 rows affected.

,seller_id,avg_rating,n_reviews,tier,updated_at
0,a61cc04793308395a840807104365121,5.00,1,gold,2026-09-29 10:27:32.094461+00:00
1,6d2f2e3b539480db1e0842b3a4e32e6e,3.67,3,silver,2026-09-29 10:27:32.094461+00:00
2,c53bcd3be457a342a97e39e5a9f0be22,3.20,5,bronze,2026-09-29 10:27:32.094461+00:00
3,9591fc341b1bfb7ef561e2968ec6e011,1.00,1,bronze,2026-09-29 10:27:32.094461+00:00
4,f9ec7093df3a7b346b7bcf7864069ca3,4.78,9,gold,2026-09-29 10:27:32.094461+00:00
...,...,...,...,...,...
3090,b2eecf5ea250510da76590ca79d60e5d,5.00,9,gold,2026-09-29 10:27:32.094461+00:00
3091,eae9af4811c294c56795d70e715b7337,5.00,1,gold,2026-09-29 10:27:32.094461+00:00
3092,1c83418358f249c9d6affa52d579db0e,5.00,1,gold,2026-09-29 10:27:32.094461+00:00
3093,845e8036c3f7bf62099352f5ed910240,3.95,21,silver,2026-09-29 10:27:32.094461+00:00


In [23]:
%%sql
SELECT
    COALESCE(tier, '(no reviews)')                       AS tier,
    COUNT(*)                                             AS n_sellers,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2)   AS pct_sellers,
    SUM(n_reviews)                                       AS total_reviews,
    ROUND(AVG(avg_rating), 3)                            AS mean_of_seller_avg,
    COUNT(DISTINCT updated_at)                           AS distinct_updated_at,
    MAX(updated_at)                                      AS updated_at
FROM seller_score
GROUP BY COALESCE(tier, '(no reviews)')
ORDER BY n_sellers DESC;

4 rows affected.

,tier,n_sellers,pct_sellers,total_reviews,mean_of_seller_avg,distinct_updated_at,updated_at
0,silver,1568,50.66,87123,4.063,1,2026-09-29 10:27:32.094461+00:00
1,gold,983,31.76,7093,4.827,1,2026-09-29 10:27:32.094461+00:00
2,bronze,539,17.42,5031,2.301,1,2026-09-29 10:27:32.094461+00:00
3,(no reviews),5,0.16,0,None,1,2026-09-29 10:27:32.094461+00:00


**Висновок.** Після другого запуску в `seller_score` як і раніше **3 095 рядків** —
жодного дубля. Значення `distinct_updated_at = 1` в обох прогонах означає, що всі рядки
оновились одним `NOW()` тієї самої транзакції, а колонка `updated_at` показує, що сам цей момент
між прогонами змінився — тобто другий запуск справді переписав рядки, а не пройшов повз них. Саме так має
поводитись ETL-крок, який щодня переобчислює вітрину: повторний запуск безпечний.

## 2.4 `UPDATE ... FROM` — прапорець запізнення в `olist_orders`

In [24]:
%%sql
ALTER TABLE olist_orders
    ADD COLUMN IF NOT EXISTS is_late BOOLEAN NOT NULL DEFAULT FALSE;

WITH delivery_flags AS (
    SELECT
        order_id,
        (
            order_delivered_customer_date IS NOT NULL
            AND order_estimated_delivery_date IS NOT NULL
            AND order_delivered_customer_date::DATE > order_estimated_delivery_date::DATE
        ) AS is_late_calc
    FROM olist_orders
)
UPDATE olist_orders AS o
SET is_late = f.is_late_calc
FROM delivery_flags AS f
WHERE f.order_id = o.order_id
RETURNING o.order_id, o.customer_id, o.order_status, o.is_late;

99441 rows affected.

,order_id,customer_id,order_status,is_late
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,False
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,False
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,False
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,False
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,False
...,...,...,...,...
99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,False
99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,False
99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,False
99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,False


Розподіл значення після оновлення:

In [25]:
%%sql
SELECT
    o.is_late,
    COUNT(*)                                             AS n_orders,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2)   AS pct_orders,
    COUNT(*) FILTER (WHERE o.order_delivered_customer_date IS NULL) AS not_delivered_yet,
    ROUND(AVG(
        o.order_delivered_customer_date::DATE - o.order_estimated_delivery_date::DATE
    ), 2)                                                AS avg_days_vs_estimate
FROM olist_orders AS o
GROUP BY o.is_late
ORDER BY o.is_late;

2 rows affected.

,is_late,n_orders,pct_orders,not_delivered_yet,avg_days_vs_estimate
0,False,92906,93.43,2965,-13.51
1,True,6535,6.57,0,10.62


**Висновок 2.4.** Оновлено всі **99 441** замовлення: **6 535 (6.57 %)** приїхали пізніше
обіцяної дати, **92 906 (93.43 %)** — ні. Важлива деталь у цих 93 %: туди входять **2 965**
замовлень, які взагалі **не доставлені** (`order_delivered_customer_date IS NULL`) — скасовані,
недоступні, ще в дорозі. `FALSE` тут означає «не зафіксовано запізнення», а не «привезли вчасно»,
і для ML це різні речі: такі рядки треба або відкидати, або кодувати окремою категорією.

Середні числа показують, наскільки обережний план доставки в Olist: вчасні замовлення приходять
у середньому за **13.5 дня до** обіцяної дати, а запізнілі — лише на **10.6 дня після**.
Тобто estimate свідомо завищений із запасом.

**Чому це ідемпотентно.** `UPDATE ... FROM delivery_flags` не дописує `TRUE` поверх попереднього
стану, а **перераховує** прапорець із поточних дат для кожного рядка. Повторний запуск дасть той
самий результат; якби дата доставки змінилась, прапорець змінився б разом із нею — зокрема міг би
й зникнути. Варіант `UPDATE ... SET is_late = TRUE WHERE ...` цієї властивості не має.

## 2.5 `DELETE ... RETURNING` — контрольоване прибирання застарілих alert-ів

Прямий `DELETE ... WHERE created_at < NOW() - INTERVAL '30 days'` на щойно створеній таблиці
повернув би порожньо: єдиному alert-у кілька секунд. Щоб показати, що робить `RETURNING`,
спершу створимо три свідомо «старі» записи з явним `created_at` (значення `DEFAULT` можна
перекрити), а потім видалимо їх.

In [26]:
%%sql
INSERT INTO seller_alerts (seller_id, alert_type, severity, details, created_at)
SELECT
    s.seller_id,
    'data_quality',
    2,
    'Backdated test alert: seller has no city recorded',
    NOW() - INTERVAL '90 days'
FROM olist_sellers AS s
ORDER BY s.seller_id
LIMIT 3
RETURNING alert_id, seller_id, alert_type, created_at;

3 rows affected.

,alert_id,seller_id,alert_type,created_at
0,2,0015a82c2db000af6aaaf3ae2ecb0532,data_quality,2026-07-01 10:29:36.015475+00:00
1,3,001cca7ae9ae17fb1caed9dfb1094831,data_quality,2026-07-01 10:29:36.015475+00:00
2,4,001e6ad469a905060d959994f1b41e4f,data_quality,2026-07-01 10:29:36.015475+00:00


In [27]:
%%sql
DELETE FROM seller_alerts
WHERE created_at < NOW() - INTERVAL '30 days'
RETURNING alert_id, seller_id, alert_type, created_at;

3 rows affected.

,alert_id,seller_id,alert_type,created_at
0,2,0015a82c2db000af6aaaf3ae2ecb0532,data_quality,2026-07-01 10:29:36.015475+00:00
1,3,001cca7ae9ae17fb1caed9dfb1094831,data_quality,2026-07-01 10:29:36.015475+00:00
2,4,001e6ad469a905060d959994f1b41e4f,data_quality,2026-07-01 10:29:36.015475+00:00


Повторний запуск того самого `DELETE`:

In [30]:
%%sql
DELETE FROM seller_alerts
WHERE created_at < NOW() - INTERVAL '30 days'
RETURNING alert_id, seller_id, alert_type, created_at;

,alert_id,seller_id,alert_type,created_at


In [31]:
%%sql
SELECT alert_id, seller_id, alert_type, severity, details, created_at
FROM seller_alerts
ORDER BY alert_id;

1 rows affected.

,alert_id,seller_id,alert_type,severity,details,created_at
0,1,4a3ca9315b744ce9f8e9374361493884,late_delivery,4,Seller has 172 late delivered orders in the loaded dataset,2026-09-29 10:25:19.262156+00:00


**Висновок 2.5.** Перший `DELETE` повернув **3 рядки** — саме ті, що були backdated на 90 днів;
у таблиці лишився один свіжий alert про запізнення. Другий, дослівно такий самий `DELETE`,
повернув **порожній результат**, і це правильна поведінка, а не помилка: рядків, старших за
30 днів, більше немає. Так і виглядає retention-задача в реальному пайплайні — вона щодня
відпрацьовує вхолосту, крім днів, коли справді є що прибрати.

`RETURNING` у `DELETE` цінний саме тим, що видалене більше нізвідки не дістати: повернуті рядки —
єдиний шанс залогувати, **що конкретно** прибрано. Порожній результат при цьому — теж повноцінний
запис в аудиті: «перевірили, видаляти не було чого».

---

# Завдання 3. `customer_segments` на рівні `customer_unique_id`

## 3.1 Чому не `customer_id`

У Olist `customer_id` — це ідентифікатор **замовлення/доставки**, а не покупця. Перевіримо, що це
означає в числах.

In [32]:
%%sql
WITH per_unique AS (
    SELECT
        c.customer_unique_id,
        COUNT(DISTINCT c.customer_id) AS n_customer_ids,
        COUNT(DISTINCT o.order_id)    AS n_orders
    FROM olist_customers AS c
    LEFT JOIN olist_orders AS o
        ON o.customer_id = c.customer_id
    GROUP BY c.customer_unique_id
)
SELECT
    (SELECT COUNT(*) FROM olist_customers)                        AS customer_id_rows,
    (SELECT COUNT(*) FROM per_unique)                             AS customer_unique_id_rows,
    (SELECT COUNT(*) FROM per_unique WHERE n_orders > 1)          AS repeat_buyers,
    (SELECT MAX(n_orders) FROM per_unique)                        AS max_orders_one_buyer,
    ROUND((SELECT AVG(n_orders) FROM per_unique), 4)              AS avg_orders_per_unique_id,
    ROUND((SELECT AVG(n_orders)
             FROM (SELECT COUNT(DISTINCT o.order_id) AS n_orders
                     FROM olist_customers AS c
                     LEFT JOIN olist_orders AS o ON o.customer_id = c.customer_id
                    GROUP BY c.customer_id) AS t), 4)             AS avg_orders_per_customer_id;

1 rows affected.

,customer_id_rows,customer_unique_id_rows,repeat_buyers,max_orders_one_buyer,avg_orders_per_unique_id,avg_orders_per_customer_id
0,99441,96096,2997,17,1.0348,1.0000


**Висновок 3.1.** **99 441 `customer_id`** відповідають лише **96 096 реальним покупцям**.
Вирішальне число — праворуч: середня кількість замовлень на `customer_id` дорівнює рівно
**1.0000**, тобто за цим ключем *кожен* покупець виглядає як одноразовий, а frequency у RFM
вироджується в константу. На рівні `customer_unique_id` середнє вже **1.0348**, і зʼявляються
**2 997 повторних покупців**, серед них рекордсмен із **17 замовленнями**.

Повторні покупки — 3 % бази, але саме вони цікаві для retention-моделей; сегментація за
`customer_id` втратила б їх повністю. Тому RFM рахуємо за `customer_unique_id`.

## 3.2 Дедуплікований customer dimension

`customer_segments` не може посилатися напряму на `olist_customers`: там `customer_unique_id`
не унікальний, а FK вимагає унікального цільового стовпця. Тому спершу потрібна довідникова
таблиця з `customer_unique_id` як `PRIMARY KEY`. `DISTINCT ON` залишає по одному рядку на покупця,
беручи атрибути з **найсвіжішого** замовлення.

In [33]:
%%sql
DROP TABLE IF EXISTS customer_segments CASCADE;
DROP TABLE IF EXISTS olist_customer_dim CASCADE;

CREATE TABLE olist_customer_dim (
    customer_unique_id TEXT PRIMARY KEY,
    latest_customer_id TEXT REFERENCES olist_customers(customer_id),
    customer_state     CHAR(2),
    customer_city      TEXT
);

INSERT INTO olist_customer_dim (
    customer_unique_id,
    latest_customer_id,
    customer_state,
    customer_city
)
SELECT DISTINCT ON (c.customer_unique_id)
    c.customer_unique_id,
    c.customer_id AS latest_customer_id,
    c.customer_state,
    c.customer_city
FROM olist_customers AS c
LEFT JOIN olist_orders AS o
    ON o.customer_id = c.customer_id
ORDER BY
    c.customer_unique_id,
    o.order_purchase_timestamp DESC NULLS LAST,
    c.customer_id;

SELECT COUNT(*) AS customer_dim_rows FROM olist_customer_dim;

96096 rows affected.

1 rows affected.

,customer_dim_rows
0,96096


## 3.3 DDL `customer_segments`

Production-подібний набір обмежень: identity-PK, `UNIQUE` на бізнес-ключі, FK на довідник,
чотири `CHECK`, `NOT NULL` і `DEFAULT`.

In [34]:
%%sql
CREATE TABLE customer_segments (
    segment_id          BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,

    customer_unique_id  TEXT NOT NULL,
    segment_name        TEXT NOT NULL
        CHECK (segment_name IN ('VIP', 'regular', 'new', 'inactive')),

    rfm_score           SMALLINT NOT NULL
        CHECK (rfm_score BETWEEN 1 AND 5),

    recency_days        INTEGER
        CHECK (recency_days IS NULL OR recency_days >= 0),

    monetary_value      NUMERIC(12, 2) NOT NULL DEFAULT 0
        CHECK (monetary_value >= 0),

    n_orders            INTEGER NOT NULL DEFAULT 0
        CHECK (n_orders >= 0),

    assigned_at         TIMESTAMPTZ NOT NULL DEFAULT NOW(),

    UNIQUE (customer_unique_id)
);

ALTER TABLE customer_segments
    ADD CONSTRAINT fk_customer_segments_customer_unique
    FOREIGN KEY (customer_unique_id)
    REFERENCES olist_customer_dim(customer_unique_id)
    ON DELETE CASCADE;

SELECT COUNT(*) AS customer_segments_rows FROM customer_segments;

1 rows affected.

,customer_segments_rows
0,0


## 3.4 Наповнення через `INSERT ... SELECT ... ON CONFLICT`

Опорна дата береться **з самого датасету**, а не з `CURRENT_DATE`.

In [35]:
%%sql
SELECT
    MAX(order_purchase_timestamp)             AS last_purchase_in_dataset,
    MAX(order_purchase_timestamp)::DATE + 1   AS reference_date,
    CURRENT_DATE                              AS today,
    CURRENT_DATE - MAX(order_purchase_timestamp)::DATE AS days_between
FROM olist_orders;

1 rows affected.

,last_purchase_in_dataset,reference_date,today,days_between
0,2018-10-17 17:30:18,2018-10-18,2026-09-29,2904


Останнє замовлення в Olist зроблено **2018-10-17**, тому `reference_date = 2018-10-18`. Якби
recency рахувався від `CURRENT_DATE`, до кожного значення додалося б **2 904 дні** (і це число
росло б щодня), усі покупці стали б однаково «давніми», а notebook давав би різні результати
при кожному запуску. Опорна дата з даних робить сегментацію відтворюваною.

In [36]:
%%sql
WITH reference_date AS (
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
order_totals AS (
    SELECT
        o.order_id,
        o.customer_id,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(oi.price + oi.freight_value), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.order_id,
        o.customer_id,
        o.order_purchase_timestamp
),
customer_stats AS (
    SELECT
        c.customer_unique_id,
        COUNT(DISTINCT ot.order_id) AS n_orders,
        COALESCE(SUM(ot.order_total), 0)::NUMERIC(12, 2) AS monetary_value,
        MAX(ot.order_date) AS last_order_date
    FROM olist_customer_dim AS cd
    JOIN olist_customers AS c
        ON c.customer_unique_id = cd.customer_unique_id
    LEFT JOIN order_totals AS ot
        ON ot.customer_id = c.customer_id
    GROUP BY c.customer_unique_id
),
segmented AS (
    SELECT
        cs.customer_unique_id,
        cs.n_orders,
        cs.monetary_value,
        CASE
            WHEN cs.last_order_date IS NULL THEN NULL
            ELSE (rd.as_of_date - cs.last_order_date)::INTEGER
        END AS recency_days,
        CASE
            WHEN cs.monetary_value >= 1000 THEN 'VIP'
            WHEN cs.last_order_date IS NOT NULL
                 AND (rd.as_of_date - cs.last_order_date)::INTEGER <= 90
                 AND cs.n_orders = 1 THEN 'new'
            WHEN cs.n_orders >= 2 OR cs.monetary_value >= 100 THEN 'regular'
            ELSE 'inactive'
        END AS segment_name,
        CASE
            WHEN cs.monetary_value >= 1000 THEN 5
            WHEN cs.monetary_value >= 500  THEN 4
            WHEN cs.n_orders >= 2          THEN 3
            WHEN cs.n_orders = 1           THEN 2
            ELSE 1
        END::SMALLINT AS rfm_score
    FROM customer_stats AS cs
    CROSS JOIN reference_date AS rd
)
INSERT INTO customer_segments (
    customer_unique_id,
    segment_name,
    rfm_score,
    recency_days,
    monetary_value,
    n_orders
)
SELECT
    customer_unique_id,
    segment_name,
    rfm_score,
    recency_days,
    monetary_value,
    n_orders
FROM segmented
ON CONFLICT (customer_unique_id) DO UPDATE
SET segment_name   = EXCLUDED.segment_name,
    rfm_score      = EXCLUDED.rfm_score,
    recency_days   = EXCLUDED.recency_days,
    monetary_value = EXCLUDED.monetary_value,
    n_orders       = EXCLUDED.n_orders,
    assigned_at    = NOW()
RETURNING
    customer_unique_id,
    segment_name,
    rfm_score,
    recency_days,
    monetary_value,
    n_orders;

96096 rows affected.

,customer_unique_id,segment_name,rfm_score,recency_days,monetary_value,n_orders
0,0000366f3b9a7992bf8c76cfdf3221e2,regular,2,161,141.90,1
1,0000b849f77a49e4a4ce2b2a4ca5be3f,inactive,2,164,27.19,1
2,0000f46a3911fa3c0805444483337064,inactive,2,587,86.22,1
3,0000f6ccb0745a6a4b88665a16c9f078,inactive,2,371,43.62,1
4,0004aac84e0df4da2b147fca70cf8255,regular,2,338,196.89,1
...,...,...,...,...,...,...
96091,fffcf5a5ff07b0908bd4e2dbc735a684,VIP,5,497,2067.42,1
96092,fffea47cd6d3cc0a88bd621562a9d061,inactive,2,312,84.58,1
96093,ffff371b4d645b6ecea244b27531430a,regular,2,618,112.46,1
96094,ffff5962728ec6157033ef9805bacc48,regular,2,169,133.69,1


Перевірка обмежень таблиці:

In [37]:
%%sql
SELECT
    conname,
    contype,
    pg_get_constraintdef(oid) AS definition
FROM pg_constraint
WHERE conrelid = 'customer_segments'::regclass
ORDER BY conname;

8 rows affected.

,conname,contype,definition
0,customer_segments_customer_unique_id_key,u,UNIQUE (customer_unique_id)
1,customer_segments_monetary_value_check,c,CHECK ((monetary_value >= (0)::numeric))
2,customer_segments_n_orders_check,c,CHECK ((n_orders >= 0))
3,customer_segments_pkey,p,PRIMARY KEY (segment_id)
4,customer_segments_recency_days_check,c,CHECK (((recency_days IS NULL) OR (recency_days >= 0)))
5,customer_segments_rfm_score_check,c,CHECK (((rfm_score >= 1) AND (rfm_score <= 5)))
6,customer_segments_segment_name_check,c,"CHECK ((segment_name = ANY (ARRAY['VIP'::text, 'regular'::text, 'n..."
7,fk_customer_segments_customer_unique,f,FOREIGN KEY (customer_unique_id) REFERENCES olist_customer_dim(cus...


Розподіл сегментів:

In [38]:
%%sql
SELECT
    segment_name,
    COUNT(*)                                            AS n_customers,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2)  AS pct_customers,
    ROUND(AVG(n_orders), 3)                             AS avg_orders,
    ROUND(AVG(monetary_value), 2)                       AS avg_monetary,
    ROUND(AVG(recency_days), 1)                         AS avg_recency_days,
    SUM(monetary_value)::NUMERIC(14, 2)                 AS total_monetary,
    ROUND(100.0 * SUM(monetary_value) / SUM(SUM(monetary_value)) OVER (), 2) AS pct_of_gmv
FROM customer_segments
GROUP BY segment_name
ORDER BY total_monetary DESC;

4 rows affected.

,segment_name,n_customers,pct_customers,avg_orders,avg_monetary,avg_recency_days,total_monetary,pct_of_gmv
0,regular,45982,47.85,1.070,224.57,307.5,10326327.75,65.18
1,inactive,40032,41.66,1.000,59.28,316.1,2373156.14,14.98
2,VIP,1191,1.24,1.114,1591.81,290.2,1895849.80,11.97
3,new,8891,9.25,1.000,140.39,72.4,1248219.55,7.88


**Висновок 3.4.** Сегментовано всіх **96 096** покупців. Картина типова для e-commerce:

| Сегмент | Покупців | Частка | Середній чек | Частка GMV |
|---|---|---|---|---|
| `regular` | 45 982 | 47.9 % | 224.57 | **65.2 %** |
| `inactive` | 40 032 | 41.7 % | 59.28 | 15.0 % |
| `new` | 8 891 | 9.3 % | 140.39 | 7.9 % |
| `VIP` | 1 191 | **1.2 %** | 1 591.81 | **12.0 %** |

**1.2 % покупців дають 12 % обороту** — класична диспропорція, заради якої й будують сегментацію.
Водночас видно межу цих правил: `avg_orders` у VIP — лише **1.114**, тобто «VIP» тут майже завжди
означає одну дорогу покупку, а не лояльність. Для retention-задачі поріг у 1 000 реалів варто було
б доповнити умовою на кількість замовлень. Сегмент `new` — єдиний зі справді свіжою recency
(**72 дні** проти 290–316 в інших), що очікувано з визначення.

Відповіді на питання методички:

* **Чому `customer_unique_id`** — доказ у 3.1: за `customer_id` frequency завжди рівно 1, і
  2 997 повторних покупців зникають.
* **Навіщо `olist_customer_dim`** — FK вимагає унікального цільового стовпця, а в
  `olist_customers` той самий `customer_unique_id` трапляється до 17 разів. Довідник дає цю
  унікальність і водночас зберігає останні відомі місто й штат покупця.
* **Чому `reference_date` з датасету** — інакше recency залежала б від дня запуску notebook
  (сьогодні це зсув у 2 904 дні), і *Restart & Run All* не відтворював би числа.
* **Як `ON CONFLICT` робить сегментацію повторюваною** — `UNIQUE (customer_unique_id)` ловить
  конфлікт, і замість другого рядка покупець отримує оновлені метрики та новий `assigned_at`.

### Демонстрація idempotency

In [39]:
%%sql
SELECT
    COUNT(*)                          AS rows_in_table,
    COUNT(DISTINCT customer_unique_id) AS distinct_customers,
    MIN(segment_id)                   AS min_segment_id,
    MAX(segment_id)                   AS max_segment_id,
    COUNT(DISTINCT assigned_at)       AS distinct_assigned_at,
    MAX(assigned_at)                  AS assigned_at
FROM customer_segments;

1 rows affected.

,rows_in_table,distinct_customers,min_segment_id,max_segment_id,distinct_assigned_at,assigned_at
0,96096,96096,1,96096,1,2026-09-29 10:37:14.478328+00:00


In [40]:
%%sql
WITH reference_date AS (
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
order_totals AS (
    SELECT
        o.order_id,
        o.customer_id,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(oi.price + oi.freight_value), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.order_id,
        o.customer_id,
        o.order_purchase_timestamp
),
customer_stats AS (
    SELECT
        c.customer_unique_id,
        COUNT(DISTINCT ot.order_id) AS n_orders,
        COALESCE(SUM(ot.order_total), 0)::NUMERIC(12, 2) AS monetary_value,
        MAX(ot.order_date) AS last_order_date
    FROM olist_customer_dim AS cd
    JOIN olist_customers AS c
        ON c.customer_unique_id = cd.customer_unique_id
    LEFT JOIN order_totals AS ot
        ON ot.customer_id = c.customer_id
    GROUP BY c.customer_unique_id
),
segmented AS (
    SELECT
        cs.customer_unique_id,
        cs.n_orders,
        cs.monetary_value,
        CASE
            WHEN cs.last_order_date IS NULL THEN NULL
            ELSE (rd.as_of_date - cs.last_order_date)::INTEGER
        END AS recency_days,
        CASE
            WHEN cs.monetary_value >= 1000 THEN 'VIP'
            WHEN cs.last_order_date IS NOT NULL
                 AND (rd.as_of_date - cs.last_order_date)::INTEGER <= 90
                 AND cs.n_orders = 1 THEN 'new'
            WHEN cs.n_orders >= 2 OR cs.monetary_value >= 100 THEN 'regular'
            ELSE 'inactive'
        END AS segment_name,
        CASE
            WHEN cs.monetary_value >= 1000 THEN 5
            WHEN cs.monetary_value >= 500  THEN 4
            WHEN cs.n_orders >= 2          THEN 3
            WHEN cs.n_orders = 1           THEN 2
            ELSE 1
        END::SMALLINT AS rfm_score
    FROM customer_stats AS cs
    CROSS JOIN reference_date AS rd
)
INSERT INTO customer_segments (
    customer_unique_id,
    segment_name,
    rfm_score,
    recency_days,
    monetary_value,
    n_orders
)
SELECT
    customer_unique_id,
    segment_name,
    rfm_score,
    recency_days,
    monetary_value,
    n_orders
FROM segmented
ON CONFLICT (customer_unique_id) DO UPDATE
SET segment_name   = EXCLUDED.segment_name,
    rfm_score      = EXCLUDED.rfm_score,
    recency_days   = EXCLUDED.recency_days,
    monetary_value = EXCLUDED.monetary_value,
    n_orders       = EXCLUDED.n_orders,
    assigned_at    = NOW()
RETURNING
    customer_unique_id,
    segment_name,
    rfm_score,
    recency_days,
    monetary_value,
    n_orders;

96096 rows affected.

,customer_unique_id,segment_name,rfm_score,recency_days,monetary_value,n_orders
0,0000366f3b9a7992bf8c76cfdf3221e2,regular,2,161,141.90,1
1,0000b849f77a49e4a4ce2b2a4ca5be3f,inactive,2,164,27.19,1
2,0000f46a3911fa3c0805444483337064,inactive,2,587,86.22,1
3,0000f6ccb0745a6a4b88665a16c9f078,inactive,2,371,43.62,1
4,0004aac84e0df4da2b147fca70cf8255,regular,2,338,196.89,1
...,...,...,...,...,...,...
96091,fffcf5a5ff07b0908bd4e2dbc735a684,VIP,5,497,2067.42,1
96092,fffea47cd6d3cc0a88bd621562a9d061,inactive,2,312,84.58,1
96093,ffff371b4d645b6ecea244b27531430a,regular,2,618,112.46,1
96094,ffff5962728ec6157033ef9805bacc48,regular,2,169,133.69,1


In [41]:
%%sql
SELECT
    COUNT(*)                          AS rows_in_table,
    COUNT(DISTINCT customer_unique_id) AS distinct_customers,
    MIN(segment_id)                   AS min_segment_id,
    MAX(segment_id)                   AS max_segment_id,
    COUNT(DISTINCT assigned_at)       AS distinct_assigned_at,
    MAX(assigned_at)                  AS assigned_at
FROM customer_segments;

1 rows affected.

,rows_in_table,distinct_customers,min_segment_id,max_segment_id,distinct_assigned_at,assigned_at
0,96096,96096,1,96096,1,2026-09-29 10:39:18.922230+00:00


**Висновок.** До й після повторного прогону — **96 096 рядків і 96 096 унікальних покупців**,
`segment_id` лишився в діапазоні 1…96 096. Тобто `ON CONFLICT DO UPDATE` не витратив жодного
нового identity-значення й не створив дублів; змінилася тільки колонка `assigned_at`, і це видно
у виводі обох клітинок.

---

# Завдання 4. JOIN-запити

Шість запитів у форматі **business-question → SQL → interpretation**. Покриття: `INNER JOIN` на
чотирьох таблицях, `LEFT JOIN` з `COALESCE`, `FULL OUTER JOIN`, `SELF JOIN` і два anti-join —
через `LEFT JOIN ... IS NULL` і через `NOT EXISTS`.

## 4.1 INNER JOIN на чотирьох таблицях

**Business-question.** Які поєднання категорії товару та штату продавця дають найбільший revenue
у доставлених замовленнях — тобто де саме зосереджені гроші майданчика?

In [42]:
%%sql
SELECT
    COALESCE(p.product_category_name, '(unknown)') AS product_category_name,
    s.seller_state,
    COUNT(DISTINCT oi.order_id)                    AS n_orders,
    SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS revenue,
    ROUND(AVG(oi.price), 2)                        AS avg_item_price
FROM olist_order_items AS oi
JOIN olist_orders AS o
    ON o.order_id = oi.order_id
JOIN olist_products AS p
    ON p.product_id = oi.product_id
JOIN olist_sellers AS s
    ON s.seller_id = oi.seller_id
WHERE o.order_status = 'delivered'
GROUP BY p.product_category_name, s.seller_state
ORDER BY revenue DESC NULLS LAST
LIMIT 10;

10 rows affected.

,product_category_name,seller_state,n_orders,revenue,avg_item_price
0,cama_mesa_banho,SP,8214,1070048.14,92.79
1,relogios_presentes,SP,4494,1016168.80,198.75
2,beleza_saude,SP,5684,790967.66,109.28
3,esporte_lazer,SP,4772,680165.54,110.18
4,moveis_decoracao,SP,4597,605650.63,79.53
5,cool_stuff,SP,2621,484196.53,156.85
6,utilidades_domesticas,SP,4021,471810.66,78.70
7,automotivo,SP,3018,438833.22,115.48
8,informatica_acessorios,SP,2783,388506.42,105.54
9,bebes,SP,1981,339129.74,139.45


**Interpretation.** Усі десять рядків топу — **штат SP (Сан-Паулу)**, без жодного винятку.
Лідер — `cama_mesa_banho` (постіль і текстиль): **1 070 048** реалів із 8 214 замовлень при
середній ціні позиції 92.79. Одразу за ним `relogios_presentes` (годинники й подарунки) —
**1 016 168** реалів, але лише з 4 494 замовлень і середньою ціною **198.75**, тобто вдвічі
дорожчим чеком.

Це дві різні бізнес-моделі в одному топі: перша заробляє обігом дешевого товару, друга — маржею
на дорогому. Для прогнозу revenue їх не можна описувати однією ознакою «категорія»; потрібна
принаймні пара «категорія × середня ціна позиції». А монополія SP означає, що `seller_state`
у моделі буде майже константою — корисніше кодувати його як «SP / не-SP» або через відстань
між штатом продавця й покупця.

`INNER JOIN` тут доречний саме тому, що питання стосується грошей: рядок без товару, продавця
чи замовлення в revenue не потрапляє за визначенням.

## 4.2 LEFT JOIN + COALESCE: customer-level features

**Business-question.** Які ознаки можна побудувати для **реального покупця**
(`customer_unique_id`), щоб жоден покупець не зник і жоден відгук не був порахований двічі?

In [43]:
%%sql
WITH order_totals AS (
    SELECT
        o.order_id,
        c.customer_unique_id,
        SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.order_id,
        c.customer_unique_id
),
review_per_order AS (
    SELECT
        rv.order_id,
        AVG(rv.review_score)::NUMERIC(3, 2) AS order_review_score
    FROM olist_order_reviews AS rv
    WHERE rv.review_score IS NOT NULL
    GROUP BY rv.order_id
),
customer_stats AS (
    SELECT
        ot.customer_unique_id,
        COUNT(DISTINCT ot.order_id) AS n_orders,
        COALESCE(SUM(ot.order_total), 0)::NUMERIC(12, 2) AS total_spend,
        AVG(rpo.order_review_score)::NUMERIC(3, 2) AS avg_review_score
    FROM order_totals AS ot
    LEFT JOIN review_per_order AS rpo
        ON rpo.order_id = ot.order_id
    GROUP BY ot.customer_unique_id
)
SELECT
    cd.customer_unique_id,
    cd.customer_state,
    COALESCE(cs.n_orders, 0)                       AS n_orders,
    COALESCE(cs.total_spend, 0)::NUMERIC(12, 2)    AS total_spend,
    COALESCE(cs.avg_review_score, 0)::NUMERIC(3, 2) AS avg_review_score,
    (cs.avg_review_score IS NULL)                  AS review_missing
FROM olist_customer_dim AS cd
LEFT JOIN customer_stats AS cs
    ON cs.customer_unique_id = cd.customer_unique_id
ORDER BY total_spend DESC NULLS LAST
LIMIT 15;

15 rows affected.

,customer_unique_id,customer_state,n_orders,total_spend,avg_review_score,review_missing
0,0a0a92112bd4c708ca5fde585afaa872,RJ,1,13664.08,1.00,False
1,da122df9eeddfedc1dc1f5349a1a690c,RJ,2,7571.63,5.00,False
2,763c8b1c9c68a0229c42c9fc6f662b93,ES,1,7274.88,1.00,False
3,dc4802a71eae9be1dd28f5d788ceb526,MS,1,6929.31,5.00,False
4,459bef486812aa25204be022145caa62,ES,1,6922.21,0.00,True
...,...,...,...,...,...,...
10,c8460e4251689ba205045f3ea17884a1,RS,4,4655.88,4.00,False
11,edde2314c6c30e864a128ac95d6b2112,SP,1,4513.32,5.00,False
12,a229eba70ec1c2abef51f04987deb7a5,RJ,1,4445.50,5.00,False
13,edf81e1f3070b9dac83ec83dacdbb9bc,DF,1,4194.76,2.00,False


Контрольний запит: скільки рядків дає кожен рівень грануляності.

In [44]:
%%sql
WITH review_per_order AS (
    SELECT rv.order_id, AVG(rv.review_score)::NUMERIC(3, 2) AS order_review_score
    FROM olist_order_reviews AS rv
    WHERE rv.review_score IS NOT NULL
    GROUP BY rv.order_id
)
SELECT
    (SELECT COUNT(*) FROM olist_order_reviews)                         AS review_rows,
    (SELECT COUNT(*) FROM review_per_order)                            AS review_per_order_rows,
    (SELECT COUNT(*)
       FROM olist_order_items AS oi
       JOIN olist_order_reviews AS rv ON rv.order_id = oi.order_id)    AS item_level_join_rows,
    (SELECT COUNT(*) FROM olist_customer_dim)                          AS customer_dim_rows;

1 rows affected.

,review_rows,review_per_order_rows,item_level_join_rows,customer_dim_rows
0,99224,98673,112372,96096


**Interpretation.** Контрольний запит показує, навіщо потрібен CTE `review_per_order`:
**99 224** рядки відгуків зводяться до **98 673** замовлень, а той самий join, зроблений на рівні
позицій, дав би **112 372** рядки. Якби `avg_review_score` рахувався без цього кроку, покупці з
багатопозиційними замовленнями отримали б рейтинг із зайвою вагою. Фінальний `LEFT JOIN` від
`olist_customer_dim` гарантує **96 096** рядків — рівно стільки, скільки покупців.

У топі витрат: покупець `0a0a9211…` з RJ витратив **13 664** реали за одне замовлення —
і поставив **1**. Те саме в `763c8b1c…` (7 275 реалів, оцінка 1) і `4007669d…` (6 082, оцінка 1).
Тобто найбільші чеки систематично отримують найгірші оцінки — це вже гіпотеза для перевірки,
а не шум.

Окремо варто дивитися на `459bef48…` (6 922 реали): у нього `review_missing = TRUE`, і
`COALESCE(..., 0)` перетворив «відгуку немає» на **0.00**. Для виводу це зручно, але як ML-ознака
небезпечно: 0 опиниться на шкалі нижче за найгіршу реальну оцінку 1. Саме тому в запиті поряд
є явний прапорець `review_missing` — у модель треба віддавати пару «значення + ознака пропуску»,
а не замаскований нуль.

## 4.3 FULL OUTER JOIN: географія клієнтів і продавців

**Business-question.** Які штати представлені тільки серед клієнтів, тільки серед продавців,
а які — в обох групах? Це питання про логістику: там, де є покупці й немає продавців, кожне
замовлення їде здалеку.

In [45]:
%%sql
WITH customer_states AS (
    SELECT customer_state AS state, COUNT(*) AS n_customers
    FROM olist_customers
    GROUP BY customer_state
), seller_states AS (
    SELECT seller_state AS state, COUNT(*) AS n_sellers
    FROM olist_sellers
    GROUP BY seller_state
)
SELECT
    COALESCE(c.state, s.state) AS state,
    c.n_customers,
    s.n_sellers,
    CASE
        WHEN c.state IS NOT NULL AND s.state IS NOT NULL THEN 'both'
        WHEN c.state IS NOT NULL THEN 'customers_only'
        ELSE 'sellers_only'
    END AS state_presence
FROM customer_states AS c
FULL OUTER JOIN seller_states AS s
    ON s.state = c.state
ORDER BY state_presence, n_customers DESC NULLS LAST, state;

27 rows affected.

,state,n_customers,n_sellers,state_presence
0,SP,41746,1849.0,both
1,RJ,12852,171.0,both
2,MG,11635,244.0,both
3,RS,5466,129.0,both
4,PR,5045,349.0,both
...,...,...,...,...
22,AC,81,1.0,both
23,AL,413,NaN,customers_only
24,TO,280,NaN,customers_only
25,AP,68,NaN,customers_only


Зведення по типах присутності:

In [46]:
%%sql
WITH customer_states AS (
    SELECT customer_state AS state FROM olist_customers GROUP BY customer_state
), seller_states AS (
    SELECT seller_state AS state FROM olist_sellers GROUP BY seller_state
)
SELECT
    CASE
        WHEN c.state IS NOT NULL AND s.state IS NOT NULL THEN 'both'
        WHEN c.state IS NOT NULL THEN 'customers_only'
        ELSE 'sellers_only'
    END AS state_presence,
    COUNT(*) AS n_states,
    STRING_AGG(COALESCE(c.state, s.state), ', ' ORDER BY COALESCE(c.state, s.state)) AS states
FROM customer_states AS c
FULL OUTER JOIN seller_states AS s
    ON s.state = c.state
GROUP BY 1
ORDER BY n_states DESC;

2 rows affected.

,state_presence,n_states,states
0,both,23,"AC, AM, BA, CE, DF, ES, GO, MA, MG, MS, MT, PA, PB, PE, PI, PR, RJ..."
1,customers_only,4,"AL, AP, RR, TO"


**Interpretation.** `FULL OUTER JOIN` дає **27 штатів**, і результат асиметричний:
**23 штати мають і клієнтів, і продавців**, **4 — тільки клієнтів** (`AL`, `AP`, `RR`, `TO`),
і **жодного «тільки продавці»**. Останнє й робить цей JOIN змістовним: ми не знали цього
заздалегідь, і саме порожня права сторона є відповіддю — кожен штат, де хтось продає, має й
покупців, а зворотне не виконується.

Концентрація вражає: **SP — 41 746 клієнтів і 1 849 продавців** (59.7 % усіх продавців), тоді як
`PA`, `MA`, `PI`, `AM`, `AC` мають по **одному** продавцю на сотні клієнтів. У `AL`, `AP`, `RR`,
`TO` немає жодного. Практичний наслідок: для цих регіонів доставка завжди міжштатна, і
`freight_value` та строк доставки там визначаються не товаром, а географією — це сильна ознака
для моделі затримок.

Ні `INNER JOIN` (втратив би 4 штати), ні `LEFT JOIN` від клієнтів (не показав би, що продавців
без клієнтів немає) на це питання не відповідають.

## 4.4 SELF JOIN: пари продавців для регіонального порівняння

**Business-question.** Які пари продавців з одного штату, але різних міст, найсильніше
розходяться за рейтингом? Це кандидати на розбір best practice всередині одного регіону.

Щоб порівняння було чесним, беремо лише продавців із **щонайменше 50 відгуками** — інакше в топ
вийдуть пари «5.00 з одного відгуку проти 1.00 з одного відгуку».

In [47]:
%%sql
SELECT
    s1.seller_id      AS seller_a,
    s2.seller_id      AS seller_b,
    s1.seller_state,
    s1.seller_city    AS city_a,
    s2.seller_city    AS city_b,
    sc1.avg_rating    AS rating_a,
    sc2.avg_rating    AS rating_b,
    ROUND(ABS(sc1.avg_rating - sc2.avg_rating), 2) AS rating_gap
FROM olist_sellers AS s1
JOIN olist_sellers AS s2
    ON s1.seller_state = s2.seller_state
   AND s1.seller_id < s2.seller_id
   AND s1.seller_city IS DISTINCT FROM s2.seller_city
JOIN seller_score AS sc1 ON sc1.seller_id = s1.seller_id
JOIN seller_score AS sc2 ON sc2.seller_id = s2.seller_id
WHERE sc1.n_reviews >= 50
  AND sc2.n_reviews >= 50
ORDER BY rating_gap DESC NULLS LAST, s1.seller_state, seller_a, seller_b
LIMIT 15;

15 rows affected.

,seller_a,seller_b,seller_state,city_a,city_b,rating_a,rating_b,rating_gap
0,1ca7077d890b907f89be8c954a02686a,d13e50eaa47b4cbe9eb81465865d8cfc,SP,santana de parnaiba,santo andre,2.33,4.82,2.49
1,1ca7077d890b907f89be8c954a02686a,d9bd94811c3338dceb4181f3dbc0c73e,SP,santana de parnaiba,sao paulo,2.33,4.80,2.47
2,1ca7077d890b907f89be8c954a02686a,d566c37fa119d5e66c4e9052e83ee4ea,SP,santana de parnaiba,sao paulo,2.33,4.69,2.36
3,116ccb1a1604bc88e4d234a8c23f33de,1ca7077d890b907f89be8c954a02686a,SP,sao bernardo do campo,santana de parnaiba,4.64,2.33,2.31
4,1bb2bdb95f4841f1bba2c0d2cd83d3c9,1ca7077d890b907f89be8c954a02686a,SP,sao paulo,santana de parnaiba,4.64,2.33,2.31
...,...,...,...,...,...,...,...,...
10,1ca7077d890b907f89be8c954a02686a,709e16e2b25c7474d980076c6bfc4806,SP,santana de parnaiba,birigui,2.33,4.52,2.19
11,1ca7077d890b907f89be8c954a02686a,725c32fa80c2faacc4fc88450d27314e,SP,santana de parnaiba,sao paulo,2.33,4.52,2.19
12,1ca7077d890b907f89be8c954a02686a,9d4db00d65d7760644ac0c14edb5fd86,SP,santana de parnaiba,sorocaba,2.33,4.52,2.19
13,1ca7077d890b907f89be8c954a02686a,d9a84e1403de8da0c3aa531d6d108ba6,SP,santana de parnaiba,sao paulo,2.33,4.52,2.19


**Interpretation.** Усі 15 пар — зі штату **SP**, бо тільки там достатньо продавців із 50+
відгуками. У центрі майже кожного рядка один і той самий продавець `1ca7077d…` із
**santana de parnaiba** та рейтингом **2.33**: він утворює найбільші розриви з
`d13e50ea…` (santo andre, **4.82**, розрив **2.49**), `d9bd9481…` (sao paulo, 4.80) і далі.

Тобто SELF JOIN тут відповів не зовсім на те питання, яке ставили: замість «пар для порівняння
практик» він знайшов **одного аномального продавця**, який тягне вниз усі пари зі своєю участю.
Це корисний результат — але він означає, що правильний наступний крок не «порівняти два міста»,
а «розібрати конкретного продавця»: у межах одного штату різниця в 2.5 бали не пояснюється
логістикою.

Умова `s1.seller_id < s2.seller_id` прибирає дзеркальні пари й самопари, а
`IS DISTINCT FROM` по місту коректно працює навіть якби місто було `NULL`
(звичайне `<>` тоді мовчки відкинуло б рядок).

## 4.5 Anti-join через `LEFT JOIN ... IS NULL`: замовлення без відгуку

**Business-question.** Які замовлення не мають жодного review-запису і чи залежить це від статусу
замовлення?

In [48]:
%%sql
SELECT
    o.order_id,
    o.customer_id,
    o.order_status,
    o.order_purchase_timestamp
FROM olist_orders AS o
LEFT JOIN olist_order_reviews AS rv
    ON rv.order_id = o.order_id
WHERE rv.review_row_id IS NULL
ORDER BY o.order_purchase_timestamp
LIMIT 15;

15 rows affected.

,order_id,customer_id,order_status,order_purchase_timestamp
0,cda873529ca7ab71f677d5ec11a40304,76c74aaff2f3f7355f46d9818ad092b8,shipped,2016-10-05 16:57:30
1,5da2004d85f2828349d887c989da2adc,95f592086150f0c95d3082aea047b816,delivered,2016-10-05 20:20:59
2,02190241f7190a1f3c7e0df95a749c6a,55c9044401ffab0c717f24405e2e042f,delivered,2016-10-07 10:52:22
3,0efd0bc268d34da3f01f4ff25a6d4335,95446917717bb58d553d107d0f1668f6,shipped,2016-10-07 15:53:31
4,c776863a93dc0740c6e7d78104b21413,1710b798ebdc5db6665c815b90a64999,delivered,2016-10-10 15:01:27
...,...,...,...,...
10,fcda3cc8210e746326864eb933641ccf,fee7cf49127308e2b5402c9984b6f1f6,delivered,2017-01-18 22:41:55
11,58a4317c14bcba43d0cf161f17e9db2b,e7f5b701d99a5eb91e57a970a75640f0,delivered,2017-01-30 11:55:23
12,ee55010a370e987f203499f435a59847,9f074c63790b0ef8b01c66fd67a97601,delivered,2017-01-30 15:17:47
13,91bcbd0102f5188ea47df5bcc83f3cad,0e63522224dc7d760139532f17a3634f,delivered,2017-01-31 02:29:51


Скільки таких замовлень у розрізі статусу:

In [49]:
%%sql
SELECT
    o.order_status,
    COUNT(*)                                                   AS n_orders,
    COUNT(*) FILTER (WHERE rv.review_row_id IS NULL)           AS orders_without_review,
    ROUND(100.0 * COUNT(*) FILTER (WHERE rv.review_row_id IS NULL)
          / COUNT(*), 2)                                       AS pct_without_review
FROM olist_orders AS o
LEFT JOIN (
    SELECT DISTINCT order_id, MIN(review_row_id) AS review_row_id
    FROM olist_order_reviews
    GROUP BY order_id
) AS rv
    ON rv.order_id = o.order_id
GROUP BY o.order_status
ORDER BY orders_without_review DESC;

8 rows affected.

,order_status,n_orders,orders_without_review,pct_without_review
0,delivered,96478,646,0.67
1,shipped,1107,75,6.78
2,canceled,625,20,3.20
3,unavailable,609,14,2.30
4,processing,301,6,1.99
5,invoiced,314,5,1.59
6,created,5,2,40.00
7,approved,2,0,0.00


**Interpretation.** Без відгуку залишилось **768 замовлень** із 99 441 — менше **0.8 %**.
Покриття відгуками в Olist надзвичайно високе, і це саме по собі варто памʼятати: відсутність
відгуку тут — рідкісна подія, а не норма.

Розподіл по статусах не випадковий. Серед `delivered` без відгуку лише **0.67 %**, а серед
`shipped` — **6.78 %**, серед `canceled` — 3.20 %, `unavailable` — 2.30 %. Логіка зрозуміла:
Olist просить оцінку після доставки, тож недоставлені замовлення часто й не встигають її отримати.
Найпоказовіший рядок — статус `created`: з 5 замовлень 2 без відгуку (**40 %**), але при такій
кількості це радше курйоз, ніж метрика.

Перші за часом рядки — жовтень 2016 і січень 2017, тобто «хвіст» без відгуків тяжіє до найранішого
періоду роботи майданчика, коли процес збору оцінок ще не влаштувався.

Anti-join через `LEFT JOIN ... WHERE rv.review_row_id IS NULL` тут читається буквально:
«приєднай відгуки і залиш тільки те, до чого нічого не приєдналось». Перевіряти треба саме
`review_row_id` — `NOT NULL`-колонку правої таблиці; по `review_id`, який сам буває порожнім,
перевірка дала б хибні спрацювання.

## 4.6 Anti-join через `NOT EXISTS`: продавці без жодної успішної доставки

**Business-question.** Скільки продавців зареєстровано, але жодного разу не довели замовлення
до статусу `delivered`? Це «мертві» акаунти каталогу.

In [50]:
%%sql
SELECT
    s.seller_state,
    COUNT(*) AS sellers_without_delivered_order,
    (SELECT COUNT(*) FROM olist_sellers AS s2
      WHERE s2.seller_state = s.seller_state) AS sellers_in_state,
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM olist_sellers AS s2
                               WHERE s2.seller_state = s.seller_state), 1) AS pct_of_state
FROM olist_sellers AS s
WHERE NOT EXISTS (
    SELECT 1
    FROM olist_order_items AS oi
    JOIN olist_orders AS o
        ON o.order_id = oi.order_id
    WHERE oi.seller_id = s.seller_id
      AND o.order_status = 'delivered'
)
GROUP BY s.seller_state
ORDER BY sellers_without_delivered_order DESC, s.seller_state;

11 rows affected.

,seller_state,sellers_without_delivered_order,sellers_in_state,pct_of_state
0,SP,80,1849,4.3
1,PR,14,349,4.0
2,MG,8,244,3.3
3,RJ,8,171,4.7
4,SC,6,190,3.2
5,RS,4,129,3.1
6,AC,1,1,100.0
7,BA,1,19,5.3
8,CE,1,13,7.7
9,ES,1,23,4.3


    Той самий підхід для товарів:

In [51]:
%%sql
SELECT
    COUNT(*)                                                AS products_never_delivered,
    COUNT(*) FILTER (WHERE p.product_category_name IS NULL) AS without_category,
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM olist_products), 2) AS pct_of_catalog,
    (SELECT COUNT(*) FROM olist_products AS p2
      WHERE NOT EXISTS (SELECT 1 FROM olist_order_items AS oi
                         WHERE oi.product_id = p2.product_id))  AS products_never_ordered
FROM olist_products AS p
WHERE NOT EXISTS (
    SELECT 1
    FROM olist_order_items AS oi
    JOIN olist_orders AS o
        ON o.order_id = oi.order_id
    WHERE oi.product_id = p.product_id
      AND o.order_status = 'delivered'
);

1 rows affected.

,products_never_delivered,without_category,pct_of_catalog,products_never_ordered
0,735,26,2.23,0


**Interpretation.** **125 продавців із 3 095 (4 %)** не мають жодного доставленого замовлення;
абсолютна більшість — у SP (**80**), бо там і продавців найбільше, а частка по штатах усюди
схожа (3–5 %). Виняток промовистий: у штаті **AC єдиний продавець — і він саме такий**, тобто
покриття цього штату номінальне, хоча у 4.3 він і рахувався як «є продавці».

З товарами картина інша: **жоден товар каталогу не лишився без замовлення** (`products_never_ordered = 0`),
але **735 товарів (2.23 %)** ніколи не дійшли до покупця в доставленому замовленні, і **26** із них
до того ж без категорії. Тобто мертвий асортимент у Olist — це не «не замовляли», а «замовили,
але не довезли»: скасування й `unavailable`.

`NOT EXISTS` тут краще за `LEFT JOIN ... IS NULL`, бо умова складніша за просту рівність
(потрібен join із `olist_orders` і фільтр по статусу): підзапит зупиняється на першому знайденому
рядку й не будує проміжний результат, який довелося б потім фільтрувати.

---

# Завдання 5. GROUP BY, HAVING і агрегати

Чотири бізнес-метрики; `HAVING` — у трьох із чотирьох; `STRING_AGG`, `ARRAY_AGG` і `JSONB_AGG` —
у 5.3 і 5.4.

## 5.1 Revenue по категоріях

**Business-question.** Які категорії дають значущий оборот і як у них співвідносяться ціна товару
та вартість доставки?

In [52]:
%%sql
SELECT
    COALESCE(p.product_category_name, '(unknown)')   AS product_category_name,
    COUNT(DISTINCT oi.order_id)                      AS n_orders,
    SUM(oi.price)::NUMERIC(12, 2)                    AS gross_revenue,
    SUM(oi.freight_value)::NUMERIC(12, 2)            AS freight,
    ROUND(AVG(oi.price), 2)                          AS avg_item_price,
    ROUND(100.0 * SUM(oi.freight_value) / NULLIF(SUM(oi.price), 0), 2) AS freight_pct_of_price
FROM olist_order_items AS oi
JOIN olist_products AS p
    ON p.product_id = oi.product_id
GROUP BY p.product_category_name
HAVING SUM(oi.price) > 10000
ORDER BY gross_revenue DESC NULLS LAST
LIMIT 15;

15 rows affected.

,product_category_name,n_orders,gross_revenue,freight,avg_item_price,freight_pct_of_price
0,beleza_saude,8836,1258681.34,182566.73,130.16,14.50
1,relogios_presentes,5624,1205005.68,100535.93,201.14,8.34
2,cama_mesa_banho,9417,1036988.68,204693.04,93.30,19.74
3,esporte_lazer,7720,988048.97,168607.51,114.34,17.06
4,informatica_acessorios,6689,911954.32,147318.08,116.51,16.15
...,...,...,...,...,...,...
10,brinquedos,3886,483946.60,77425.95,117.55,16.00
11,bebes,2885,411764.89,68353.11,134.34,16.60
12,perfumaria,3162,399124.87,54213.84,116.74,13.58
13,telefonia,4199,323667.53,71215.79,71.21,22.00


Скільки категорій відсіяв `HAVING`:

In [53]:
%%sql
WITH per_category AS (
    SELECT
        p.product_category_name,
        SUM(oi.price) AS gross_revenue
    FROM olist_order_items AS oi
    JOIN olist_products AS p ON p.product_id = oi.product_id
    GROUP BY p.product_category_name
)
SELECT
    COUNT(*)                                          AS categories_total,
    COUNT(*) FILTER (WHERE gross_revenue > 10000)     AS categories_kept_by_having,
    ROUND(100.0 * SUM(gross_revenue) FILTER (WHERE gross_revenue > 10000)
          / SUM(gross_revenue), 2)                    AS pct_revenue_kept
FROM per_category;

1 rows affected.

,categories_total,categories_kept_by_having,pct_revenue_kept
0,74,53,99.43


**Interpretation.** `HAVING SUM(oi.price) > 10000` залишив **53 категорії з 74**, і ці 53
покривають **99.43 % обороту** — тобто 21 відкинута категорія разом важить менше 0.6 %.
Це і є сенс `HAVING`: прибрати довгий хвіст, не втративши суті.

Топ очолює `beleza_saude` (краса й здоровʼя) — **1 258 681** реал, далі `relogios_presentes`
(**1 205 005**) і `cama_mesa_banho` (**1 036 988**).

Найцікавіша колонка — частка доставки в ціні. Вона змінюється **втричі**: `relogios_presentes` —
**8.34 %** (годинник маленький і дорогий), а `moveis_escritorio` — **25.03 %**,
`moveis_decoracao` — 23.67 %, `utilidades_domesticas` — 23.12 %, `telefonia` — **22.00 %**
(дешевий товар, тому доставка зʼїдає чверть чека). Для юніт-економіки це означає, що однакова
знижка в різних категоріях дає зовсім різний результат, а для ML — що `freight_value` треба
нормувати на ціну, а не брати абсолютним числом.

## 5.2 AOV по штатах клієнтів — і різниця між `WHERE` і `HAVING`

**Business-question.** У яких штатах середній чек доставленого замовлення найвищий і де
зосереджений оборот?

In [54]:
%%sql
WITH order_totals AS (
    SELECT
        o.order_id,
        c.customer_state,
        SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
    JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY
        o.order_id,
        c.customer_state
)
SELECT
    customer_state,
    COUNT(*)                                            AS n_orders,
    SUM(order_total)::NUMERIC(12, 2)                    AS gmv,
    ROUND(SUM(order_total) / NULLIF(COUNT(*), 0), 2)    AS aov,
    ROUND(100.0 * SUM(order_total) / SUM(SUM(order_total)) OVER (), 2) AS pct_of_gmv
FROM order_totals
GROUP BY customer_state
HAVING COUNT(*) >= 100
ORDER BY aov DESC NULLS LAST;

24 rows affected.

,customer_state,n_orders,gmv,aov,pct_of_gmv
0,PB,517,137838.55,266.61,0.90
1,AL,397,94172.49,237.21,0.61
2,RO,243,56966.00,234.43,0.37
3,PA,946,212023.57,224.13,1.38
4,PI,476,105178.19,220.96,0.68
...,...,...,...,...,...
19,RS,5345,861472.79,161.17,5.60
20,MG,11354,1818891.67,160.20,11.83
21,ES,1995,317657.93,159.23,2.07
22,PR,4923,781708.80,158.79,5.08


**Interpretation.** Цей запит — зручна ілюстрація різниці між двома фільтрами:

* **`WHERE o.order_status = 'delivered'`** працює **до** агрегації, на окремих рядках: скасовані
  й недоставлені замовлення просто не потрапляють у суму. Написати це в `HAVING` було б
  неможливо — після `GROUP BY` окремого статусу вже не існує.
* **`HAVING COUNT(*) >= 100`** працює **після** агрегації, над готовими групами: штати з
  менш ніж сотнею замовлень (`AP`, `RR`, `AC`) відкинуто, бо їхній AOV порахований на такій
  вибірці — це шум. Записати це в `WHERE` не можна: `COUNT(*)` на момент `WHERE` ще не існує.

Результат перевертає звичну інтуїцію. Найвищий середній чек — у **PB (266.61)**, `AL` (237.21),
`RO` (234.43), `PA` (224.13), тобто в невеликих північних і північно-східних штатах. Найнижчий —
у **SP: 142.46**, при тому що SP дає **37.5 % усього GMV** і 40 501 замовлення з 24 штатів у вибірці.

Пояснення радше логістичне, ніж купівельне: у віддалених штатах у чек входить дорога міжштатна
доставка (див. 4.3 — там майже немає локальних продавців), і дрібні покупки там просто не роблять.
У SP товар часто їде всередині штату, тому дешеві замовлення економічно виправдані. Висновок для
бізнесу: «високий AOV» у PB — не ознака багатшого покупця, а ознака дорожчої логістики.

## 5.3 `STRING_AGG`: покупці з найширшим кошиком категорій

**Business-question.** Які реальні покупці купували в найбільшій кількості різних категорій і що
саме вони брали? Це кандидати на cross-sell.

In [55]:
%%sql
SELECT
    c.customer_unique_id,
    COUNT(DISTINCT o.order_id)                        AS n_orders,
    COUNT(DISTINCT p.product_category_name)           AS n_categories,
    SUM(oi.price)::NUMERIC(12, 2)                     AS total_spend,
    STRING_AGG(
        DISTINCT COALESCE(p.product_category_name, '(unknown)'),
        ', ' ORDER BY COALESCE(p.product_category_name, '(unknown)')
    ) AS categories
FROM olist_customers AS c
JOIN olist_orders AS o
    ON o.customer_id = c.customer_id
JOIN olist_order_items AS oi
    ON oi.order_id = o.order_id
LEFT JOIN olist_products AS p
    ON p.product_id = oi.product_id
GROUP BY c.customer_unique_id
HAVING COUNT(DISTINCT p.product_category_name) >= 4
ORDER BY n_categories DESC, total_spend DESC
LIMIT 15;

15 rows affected.

,customer_unique_id,n_orders,n_categories,total_spend,categories
0,3e43e6105506432c953e165fb2acf44c,9,5,1000.85,"cama_mesa_banho, casa_construcao, informatica_acessorios, moveis_d..."
1,47c1a3033b8b77b3ab6e109eb4d5fdf3,6,5,836.60,"automotivo, beleza_saude, cama_mesa_banho, cool_stuff, tablets_imp..."
2,1b6c7548a2a1f9037c1fd3ddfed95f33,7,5,809.21,"beleza_saude, moveis_decoracao, papelaria, perfumaria, utilidades_..."
3,35ecdf6858edc6427223b64804cf028e,5,5,393.86,"alimentos_bebidas, beleza_saude, esporte_lazer, malas_acessorios, ..."
4,74cb1ad7e6d5674325c1f99b5ea30d82,5,5,334.50,"climatizacao, construcao_ferramentas_construcao, eletrodomesticos,..."
...,...,...,...,...,...
10,4e65032f1f574189fb793bac5a867bbc,5,4,404.57,"alimentos_bebidas, automotivo, consoles_games, ferramentas_jardim"
11,a0abd26659c9914690cbd3320d39fd96,4,4,327.19,"bebidas, esporte_lazer, la_cuisine, papelaria"
12,f8bc1d81f395f6239a2a5ab1b9edf1ab,3,4,318.54,"cama_mesa_banho, ferramentas_jardim, moveis_cozinha_area_de_servic..."
13,795c1622cf7a53d63d324e862349d01c,4,4,281.20,"bebes, esporte_lazer, papelaria, utilidades_domesticas"


**Interpretation.** Максимум по всій базі — **5 різних категорій** на покупця, і таких лише
кілька. Лідер `3e43e610…`: **9 замовлень, 5 категорій, 1 000.85** реала — постіль, будматеріали,
компʼютерні аксесуари, декор і побутові товари. Схоже на людину, яка облаштовувала житло, а не
на регулярного покупця майданчика.

`HAVING COUNT(DISTINCT p.product_category_name) >= 4` тут обовʼязковий: без нього запит повернув
би всі 96 тис. покупців, більшість із однією категорією. Фільтр за агрегатом — єдиний спосіб
відібрати групи за їхньою *внутрішньою різноманітністю*.

Рядок `63cfc61c…` показує, навіщо `COALESCE(p.product_category_name, '(unknown)')` усередині
`STRING_AGG`: у нього серед категорій є `(unknown)` — товар без картки (610 таких у каталозі).
Без `COALESCE` `STRING_AGG` мовчки пропустив би `NULL`, і список виглядав би повним, хоча насправді
частина покупки в ньому не відображена.

Практичний висновок: база Olist — це майже виключно **одноразові покупки в одній категорії**,
тому cross-sell-модель тут будувати нема на чому; реалістичніша задача — прогноз повторного
замовлення взагалі.

## 5.4 `ARRAY_AGG` + `JSONB_AGG`: профіль продавців по штатах

**Business-question.** Як виглядає якість продавців у розрізі штату і хто в кожному штаті
найпомітніший за кількістю відгуків?

In [56]:
%%sql
WITH ranked AS (
    SELECT
        s.seller_state,
        sc.seller_id,
        sc.avg_rating,
        sc.n_reviews,
        sc.tier,
        ROW_NUMBER() OVER (
            PARTITION BY s.seller_state
            ORDER BY sc.n_reviews DESC, sc.seller_id
        ) AS rn
    FROM seller_score AS sc
    JOIN olist_sellers AS s
        ON s.seller_id = sc.seller_id
    WHERE sc.avg_rating IS NOT NULL
)
SELECT
    seller_state,
    COUNT(*)                                        AS n_sellers,
    ROUND(AVG(avg_rating), 3)                       AS state_avg_rating,
    SUM(n_reviews)                                  AS total_reviews,
    ARRAY_AGG(DISTINCT tier ORDER BY tier)          AS tiers_present,
    JSONB_AGG(
        JSONB_BUILD_OBJECT(
            'seller_id', LEFT(seller_id, 8),
            'reviews',   n_reviews,
            'rating',    avg_rating,
            'tier',      tier
        )
        ORDER BY n_reviews DESC
    ) FILTER (WHERE rn <= 3)                        AS top3_sellers
FROM ranked
GROUP BY seller_state
HAVING COUNT(*) >= 20
ORDER BY state_avg_rating DESC, n_sellers DESC;

9 rows affected.

,seller_state,n_sellers,state_avg_rating,total_reviews,tiers_present,top3_sellers
0,ES,23,4.263,314,"[bronze, gold, silver]","[{'tier': 'silver', 'rating': 3.98, 'reviews': 197, 'seller_id': '..."
1,GO,40,4.114,460,"[bronze, gold, silver]","[{'tier': 'silver', 'rating': 4.42, 'reviews': 67, 'seller_id': '9..."
2,SC,190,4.077,3649,"[bronze, gold, silver]","[{'tier': 'silver', 'rating': 4.39, 'reviews': 251, 'seller_id': '..."
3,RS,129,4.057,1973,"[bronze, gold, silver]","[{'tier': 'silver', 'rating': 4.41, 'reviews': 304, 'seller_id': '..."
4,MG,244,4.054,7884,"[bronze, gold, silver]","[{'tier': 'silver', 'rating': 4.22, 'reviews': 715, 'seller_id': '..."
5,PR,347,4.040,7658,"[bronze, gold, silver]","[{'tier': 'silver', 'rating': 4.19, 'reviews': 527, 'seller_id': '..."
6,DF,30,4.032,823,"[bronze, gold, silver]","[{'tier': 'silver', 'rating': 3.87, 'reviews': 174, 'seller_id': '..."
7,RJ,171,3.977,4320,"[bronze, gold, silver]","[{'tier': 'silver', 'rating': 4.25, 'reviews': 733, 'seller_id': '..."
8,SP,1846,3.970,70399,"[bronze, gold, silver]","[{'tier': 'silver', 'rating': 3.94, 'reviews': 1838, 'seller_id': ..."


**Interpretation.** `HAVING COUNT(*) >= 20` залишив **9 штатів**, де взагалі є статистика.
І тут проявляється контрінтуїтивна закономірність: найвищий середній рейтинг — у **ES (4.263)**
із 23 продавцями, далі `GO` (4.114), `SC` (4.077), а **найнижчий — у SP (3.970)**, де продавців
**1 846** і **70 399** відгуків.

Це не означає, що в Сан-Паулу продають гірше. Це ефект розміру вибірки, той самий, що й у 2.3:
у штаті з двома десятками продавців кілька дрібних акаунтів із рейтингом 5.00 помітно піднімають
середнє, а в SP закон великих чисел стягує його до реального рівня. `ARRAY_AGG(DISTINCT tier)`
це підтверджує: у **кожному** штаті присутні всі три tier, тобто штат не визначає якість.

`JSONB_AGG` з `FILTER (WHERE rn <= 3)` дає готовий вкладений обʼєкт із трьома найпомітнішими
продавцями штату — зручний формат саме тоді, коли результат далі йде в API чи дашборд, а не
в плоску таблицю. У SP у трійці — `6560211a…` (1 838 відгуків, 3.94) і вже знайомий
`4a3ca931…` (1 785 відгуків, 3.83) — той самий продавець, якому в 2.2 виписано alert за
172 запізнення.

---

# Завдання 6. Set operations, window-функції, автоперевірки й Reflection

## 6.1 Set operations: географія клієнтів проти географії продавців

**Business-question.** У яких штатах ринок двосторонній, а в яких Olist присутній лише як
покупецький попит?

Для першої частини потрібен саме **`INTERSECT`**: питання звучить як «множина A ∩ множина B»,
і `INTERSECT` виражає це одним рядком, автоматично прибираючи дублікати з обох боків.
`INNER JOIN` дав би той самий перелік, але вимагав би `DISTINCT` і виглядав би як звʼязування
таблиць, хоча ніякого звʼязку між клієнтом і продавцем тут немає — порівнюються два незалежні
набори кодів.

In [57]:
%%sql
SELECT customer_state AS state
FROM olist_customers

INTERSECT

SELECT seller_state AS state
FROM olist_sellers
ORDER BY state;

23 rows affected.

,state
0,AC
1,AM
2,BA
3,CE
4,DF
...,...
18,RO
19,RS
20,SC
21,SE


Далі **`EXCEPT`** — асиметрична різниця, «є серед клієнтів, немає серед продавців»:

In [58]:
%%sql
SELECT customer_state AS state
FROM olist_customers

EXCEPT

SELECT seller_state AS state
FROM olist_sellers
ORDER BY state;

4 rows affected.

,state
0,AL
1,AP
2,RR
3,TO


І дзеркальний напрямок, щоб переконатися, що різниця справді однобока:

In [59]:
%%sql
SELECT seller_state AS state
FROM olist_sellers

EXCEPT

SELECT customer_state AS state
FROM olist_customers
ORDER BY state;

,state


**Interpretation.** `INTERSECT` дає **23 штати** з двостороннім ринком. `EXCEPT` у напрямку
«клієнти мінус продавці» — **4 штати**: `AL`, `AP`, `RR`, `TO`. Зворотний `EXCEPT` —
**порожній результат**, і це найважливіший рядок усього розділу: множина штатів продавців
**повністю вкладена** в множину штатів клієнтів.

Обидва оператори прибирають дублікати й повертають саме множини — тому 99 441 рядок клієнтів
і 3 095 рядків продавців стискаються до 27 унікальних кодів без жодного `GROUP BY`.
`UNION ALL` тут був би помилкою: він зберіг би всі 102 536 рядків.

Ці числа збігаються з `FULL OUTER JOIN` у 4.3 — і це корисна перехресна перевірка: два різні
інструменти дали однакову відповідь.

## 6.2 Window-функція `ROW_NUMBER()`: останнє замовлення кожного покупця

**Business-question.** Яким було найсвіжіше замовлення кожного реального покупця — і хто з них
замовляв найчастіше?

In [60]:
%%sql
WITH ranked_orders AS (
    SELECT
        c.customer_unique_id,
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp,
        ROW_NUMBER() OVER (
            PARTITION BY c.customer_unique_id
            ORDER BY o.order_purchase_timestamp DESC, o.order_id DESC
        ) AS rn,
        COUNT(*) OVER (PARTITION BY c.customer_unique_id) AS total_orders
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
)
SELECT
    customer_unique_id,
    customer_id,
    order_id,
    order_status,
    order_purchase_timestamp,
    total_orders
FROM ranked_orders
WHERE rn = 1
  AND total_orders >= 3
ORDER BY total_orders DESC, customer_unique_id
LIMIT 15;

15 rows affected.

,customer_unique_id,customer_id,order_id,order_status,order_purchase_timestamp,total_orders
0,8d50f5eadf50201ccdcedfb9e2ac8455,a682769c4bc10fc6ef2101337a6c83c9,d3582fd5ccccd9cb229a63dfb417c86f,delivered,2018-08-20 19:14:26,17
1,3e43e6105506432c953e165fb2acf44c,1815fdaaae692479110428598596b5e5,ff89ef7b3952bba5ac06d61c4a79ffbe,delivered,2018-02-27 18:36:39,9
2,1b6c7548a2a1f9037c1fd3ddfed95f33,fff7466a253c0e59499ea943462c10f9,43f08913407cac4e74a898d968e58c1a,delivered,2018-02-14 13:22:12,7
3,6469f99c1f9dfae7733b25662e7f1782,4478700ff008d3fffa02e95d89b9795f,ccb1046449f1a494de800567e7cb5979,delivered,2018-06-28 00:43:34,7
4,ca77025e7201e3b30c44b472ff346268,6ccedfba5919d72fcc8c51bfa982de62,8dacd1da7c4dc40d5c59e7ddd93b3ac8,delivered,2018-06-01 11:38:29,7
...,...,...,...,...,...,...
10,f0e310a6839dce9de1638e0fe5ab282a,e1b7e606f7b8f4064b028acf1e961c19,f2270ae03c6578b9cb6e4bba286bc962,delivered,2018-04-05 09:04:45,6
11,35ecdf6858edc6427223b64804cf028e,e6cf45b6775e86a43908d40341b3d53d,a383762bd79949e1b34878460042cd23,delivered,2018-03-31 21:18:49,5
12,394ac4de8f3acb14253c177f0e15bc58,ba7df02cddb36d302d9a87ed492fdf07,3a4e65de6ffd941f6a4ee824d472aabb,delivered,2018-08-15 18:01:23,5
13,4e65032f1f574189fb793bac5a867bbc,21e44f906e88b44ed1ada6b8d03bdc8e,1c020c2a20a524371e81f9ef21eca0b6,delivered,2018-06-14 09:28:49,5


**Interpretation.** `ROW_NUMBER() OVER (PARTITION BY customer_unique_id ORDER BY
order_purchase_timestamp DESC)` нумерує замовлення всередині кожного покупця, і `WHERE rn = 1`
залишає найсвіжіше. Друга віконна функція, `COUNT(*) OVER (PARTITION BY ...)`, додає загальну
кількість замовлень **без окремого `GROUP BY`** — це і є головна перевага window-функцій:
агрегат рахується поруч із детальним рядком, а не замість нього.

Рекордсмен — `8d50f5ea…` із **17 замовленнями**, останнє з яких у серпні 2018. Далі різкий обрив:
9, 7, 7, 7, 6… Тобто «часті покупці» Olist — це одиниці людей, і хвіст закінчується майже одразу.

Альтернатива без window-функцій — `GROUP BY customer_unique_id` з `MAX(timestamp)` і повторний
join назад до замовлень, щоб дістати `order_id`. Це два проходи по таблиці замість одного і
зайвий join, який ще й зламається при однаковому часі двох замовлень.

## 6.3 Window-функція `RANK()` і концентрація обороту

**Business-question.** Наскільки оборот сконцентрований на кількох продавцях?

In [61]:
%%sql
WITH seller_gmv AS (
    SELECT
        oi.seller_id,
        COUNT(DISTINCT oi.order_id)                      AS n_orders,
        SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS gmv
    FROM olist_order_items AS oi
    JOIN olist_orders AS o
        ON o.order_id = oi.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY oi.seller_id
)
SELECT
    RANK() OVER (ORDER BY g.gmv DESC)                   AS gmv_rank,
    LEFT(g.seller_id, 8)                                AS seller_id,
    s.seller_state,
    g.n_orders,
    g.gmv,
    sc.avg_rating,
    ROUND(100.0 * g.gmv / SUM(g.gmv) OVER (), 3)        AS pct_of_gmv,
    ROUND(100.0 * SUM(g.gmv) OVER (
              ORDER BY g.gmv DESC
              ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
          ) / SUM(g.gmv) OVER (), 2)                    AS cumulative_pct
FROM seller_gmv AS g
JOIN olist_sellers AS s
    ON s.seller_id = g.seller_id
LEFT JOIN seller_score AS sc
    ON sc.seller_id = g.seller_id
ORDER BY gmv_rank
LIMIT 15;

15 rows affected.

,gmv_rank,seller_id,seller_state,n_orders,gmv,avg_rating,pct_of_gmv,cumulative_pct
0,1,4869f7a5,SP,1124,247007.06,4.13,1.602,1.60
1,2,7c67e144,SP,973,237806.69,3.49,1.542,3.14
2,3,4a3ca931,SP,1772,231220.43,3.83,1.500,4.64
3,4,53243585,BA,348,230797.02,4.13,1.497,6.14
4,5,fa1c13f2,SP,578,200833.50,4.34,1.302,7.44
...,...,...,...,...,...,...,...,...
10,11,6560211a,SP,1819,148050.94,3.94,0.960,13.89
11,12,1f50f920,SP,1399,141712.94,4.13,0.919,14.81
12,13,46dc3b2c,RJ,503,134162.88,4.19,0.870,15.68
13,14,a1043baf,MG,702,130412.74,4.22,0.846,16.52


Скільки продавців формують половину й 80 % обороту:

In [62]:
%%sql
WITH seller_gmv AS (
    SELECT
        oi.seller_id,
        SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS gmv
    FROM olist_order_items AS oi
    JOIN olist_orders AS o ON o.order_id = oi.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY oi.seller_id
),
cumulative AS (
    SELECT
        seller_id,
        gmv,
        ROW_NUMBER() OVER (ORDER BY gmv DESC) AS rn,
        SUM(gmv) OVER (ORDER BY gmv DESC ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
            / SUM(gmv) OVER () AS cum_share
    FROM seller_gmv
)
SELECT
    COUNT(*)                                                       AS sellers_with_delivered_gmv,
    MIN(rn) FILTER (WHERE cum_share >= 0.50)                       AS sellers_for_50pct_gmv,
    MIN(rn) FILTER (WHERE cum_share >= 0.80)                       AS sellers_for_80pct_gmv,
    ROUND(100.0 * MIN(rn) FILTER (WHERE cum_share >= 0.80) / COUNT(*), 2) AS pct_sellers_for_80pct_gmv,
    SUM(gmv)::NUMERIC(14, 2)                                       AS total_delivered_gmv
FROM cumulative;

1 rows affected.

,sellers_with_delivered_gmv,sellers_for_50pct_gmv,sellers_for_80pct_gmv,pct_sellers_for_80pct_gmv,total_delivered_gmv
0,2970,133,551,18.55,15419773.75


**Interpretation.** `RANK() OVER (ORDER BY gmv DESC)` дає позицію, `SUM(gmv) OVER ()` —
знаменник для частки, а `SUM(gmv) OVER (ORDER BY gmv DESC ROWS BETWEEN UNBOUNDED PRECEDING
AND CURRENT ROW)` — накопичену суму. Три різні вікна в одному запиті, жодного підзапиту.

Розподіл виявився **менш концентрованим**, ніж підказує правило Парето: найбільший продавець
`4869f7a5…` дає лише **1.6 %** обороту, уся топ-15 разом — **17.3 %**, а щоб набрати 80 % GMV,
потрібно **551 продавця з 2 970 (18.6 %)**. Половину обороту дають **133** продавці.
Тобто це не «20/80», а радше «19/80» — і платформа значно менш залежна від окремого продавця,
ніж здається.

Окремо варто подивитись на 3-й рядок: `4a3ca931…` має **1 772 замовлення** — більше за всіх
у топі — але за GMV лише третій, а рейтинг у нього **3.83**. Це той самий продавець, якому в 2.2
виписано alert за **172 запізнення**, і той самий, що потрапив у трійку SP у 5.4. Три різні
запити зійшлися на одному акаунті: великий обіг дешевого товару, слабкий сервіс. Саме такий
профіль і має підсвічувати seller-скоринг.

## 6.4 Автоматичні перевірки

Клітинка нижче перевіряє інваріанти, на яких тримається вся робота, і падає з `AssertionError`,
якщо щось поїхало. Це та перевірка, яку в реальному пайплайні робив би окремий крок якості даних.

In [63]:
from IPython import get_ipython

ip = get_ipython()


def q(sql_text):
    """Виконує SQL і повертає результат як DataFrame."""
    return ip.run_cell_magic("sql", "", sql_text)


# --- 1. raw і typed рівні збігаються рядок у рядок -----------------------
counts = q("""
    SELECT
        (SELECT COUNT(*) FROM olist_customers_raw)    AS raw_customers,
        (SELECT COUNT(*) FROM olist_customers)        AS typed_customers,
        (SELECT COUNT(*) FROM olist_orders_raw)       AS raw_orders,
        (SELECT COUNT(*) FROM olist_orders)           AS typed_orders,
        (SELECT COUNT(*) FROM olist_order_items_raw)  AS raw_items,
        (SELECT COUNT(*) FROM olist_order_items)      AS typed_items,
        (SELECT COUNT(*) FROM olist_order_reviews_raw) AS raw_reviews,
        (SELECT COUNT(*) FROM olist_order_reviews)    AS typed_reviews;
""").iloc[0]
for a, b in [("customers", "customers"), ("orders", "orders"),
             ("items", "items"), ("reviews", "reviews")]:
    assert counts[f"raw_{a}"] == counts[f"typed_{b}"], f"raw != typed для {a}"

# --- 2. seller_score не роздутий item-level join-ом ----------------------
grain = q("""
    WITH seller_orders AS (
        SELECT DISTINCT seller_id, order_id FROM olist_order_items
    ),
    review_per_order AS (
        SELECT order_id FROM olist_order_reviews
        WHERE review_score IS NOT NULL GROUP BY order_id
    )
    SELECT
        (SELECT SUM(n_reviews) FROM seller_score)                  AS score_reviews,
        (SELECT COUNT(*) FROM seller_orders so
          JOIN review_per_order rpo ON rpo.order_id = so.order_id) AS expected_reviews,
        (SELECT COUNT(*) FROM olist_order_items oi
          JOIN olist_order_reviews rv ON rv.order_id = oi.order_id) AS naive_reviews;
""").iloc[0]
assert grain["score_reviews"] == grain["expected_reviews"], "seller_score має не ту грануляність"
assert grain["score_reviews"] < grain["naive_reviews"], "seller_score роздутий item-level join-ом"

# --- 3. customer_segments — рівно один рядок на реального покупця --------
seg = q("""
    SELECT
        (SELECT COUNT(*) FROM customer_segments)                        AS segment_rows,
        (SELECT COUNT(DISTINCT customer_unique_id) FROM customer_segments) AS distinct_customers,
        (SELECT COUNT(*) FROM olist_customer_dim)                       AS dim_rows,
        (SELECT COUNT(DISTINCT customer_unique_id) FROM olist_customers) AS source_customers;
""").iloc[0]
assert seg["segment_rows"] == seg["distinct_customers"] == seg["dim_rows"] == seg["source_customers"], \
    "customer_segments не на рівні customer_unique_id"

# --- 4. обмеження справді існують ----------------------------------------
cons = q("""
    SELECT
        COUNT(*) FILTER (WHERE contype = 'p') AS pk,
        COUNT(*) FILTER (WHERE contype = 'f') AS fk,
        COUNT(*) FILTER (WHERE contype = 'u') AS uniq,
        COUNT(*) FILTER (WHERE contype = 'c') AS checks
    FROM pg_constraint con
    JOIN pg_class c ON c.oid = con.conrelid
    JOIN pg_namespace n ON n.oid = c.relnamespace
    WHERE n.nspname = 'public'
      AND c.relname IN ('olist_customers', 'olist_orders', 'olist_order_items',
                        'olist_products', 'olist_sellers', 'olist_order_reviews',
                        'olist_customer_dim', 'customer_segments',
                        'seller_score', 'seller_alerts');
""").iloc[0]
assert cons["pk"] >= 10 and cons["fk"] >= 8 and cons["uniq"] >= 1 and cons["checks"] >= 15, \
    f"замало обмежень: {dict(cons)}"

# --- 5. жоден рядок не порушує оголошених діапазонів ----------------------
bad = q("""
    SELECT
        (SELECT COUNT(*) FROM olist_order_reviews
          WHERE review_score IS NOT NULL AND review_score NOT BETWEEN 1 AND 5) AS bad_scores,
        (SELECT COUNT(*) FROM olist_order_items WHERE price < 0
                                                   OR freight_value < 0)       AS bad_money,
        (SELECT COUNT(*) FROM customer_segments WHERE rfm_score NOT BETWEEN 1 AND 5
                                                   OR monetary_value < 0)      AS bad_segments,
        (SELECT COUNT(*) FROM olist_orders o
          WHERE NOT EXISTS (SELECT 1 FROM olist_customers c
                             WHERE c.customer_id = o.customer_id))             AS orphan_orders;
""").iloc[0]
assert bad.sum() == 0, f"порушення обмежень: {dict(bad)}"

# --- 6. is_late узгоджений із датами --------------------------------------
late = q("""
    SELECT COUNT(*) AS mismatched
    FROM olist_orders
    WHERE is_late IS DISTINCT FROM (
        order_delivered_customer_date IS NOT NULL
        AND order_estimated_delivery_date IS NOT NULL
        AND order_delivered_customer_date::DATE > order_estimated_delivery_date::DATE
    );
""").iloc[0]["mismatched"]
assert late == 0, f"is_late розійшовся з датами у {late} рядках"

print("Усі перевірки пройдено.")
print(f"  raw == typed:              {int(counts['typed_orders']):,} замовлень, "
      f"{int(counts['typed_items']):,} позицій")
print(f"  seller_score reviews:      {int(grain['score_reviews']):,} "
      f"(наївний join дав би {int(grain['naive_reviews']):,})")
print(f"  customer_segments:         {int(seg['segment_rows']):,} покупців")
print(f"  обмежень PK/FK/UNIQUE/CHECK: {int(cons['pk'])}/{int(cons['fk'])}/"
      f"{int(cons['uniq'])}/{int(cons['checks'])}")

1 rows affected.

1 rows affected.

1 rows affected.

1 rows affected.

1 rows affected.

1 rows affected.

Усі перевірки пройдено.
  raw == typed:              99,441 замовлень, 112,650 позицій
  seller_score reviews:      99,247 (наївний join дав би 112,372)
  customer_segments:         96,096 покупців
  обмежень PK/FK/UNIQUE/CHECK: 10/9/1/20


## 6.5 Reflection

**Найскладніший JOIN — `FULL OUTER JOIN`.** Не синтаксисом, а тим, що обидві сторони можуть бути
`NULL`, тому ключ доводиться відновлювати через `COALESCE(c.state, s.state)`, а «звідки прийшов
рядок» — через `CASE`. Найважче було сформулювати як business-question запит 4.4 (SELF JOIN):
технічно пари продавців будуються легко, але питання «навіщо» зʼявилось лише після того, як я
додав поріг у 50 відгуків — без нього результат був статистичним сміттям із рейтингів по одному
відгуку.

**Найкорисніша фіча для production-ETL — `ON CONFLICT DO UPDATE`.** `RETURNING` зручний для
аудиту, constraints рятують від тихого сміття, але саме upsert робить крок пайплайну безпечним
для повторного запуску: після збою можна просто запустити ще раз. Без нього кожен retry вимагав
би `DELETE` + `INSERT` у транзакції.

**Window-функції в ML-проєкті** я б використав для побудови історичних ознак без leakage:
`ROW_NUMBER()` — щоб брати N-те замовлення покупця, `LAG()` — інтервал між покупками,
ковзні агрегати `AVG(...) OVER (PARTITION BY seller_id ORDER BY date ROWS ...)` — рейтинг
продавця **станом на момент замовлення**, а не фінальний.

**Data quality.** 814 дублікатів `review_id` і 547 замовлень із кількома відгуками; 610 товарів
без категорії та розмірів; 2 965 замовлень без дати доставки; 775 замовлень без жодної позиції;
125 продавців без жодної успішної доставки.

**Чому `customer_unique_id`.** За `customer_id` середня кількість замовлень дорівнює рівно 1.0000,
frequency вироджується, і 2 997 повторних покупців зникають.

**Чому грануляність `seller_id × order_id`.** Відгук привʼязаний до замовлення, а позиції —
дрібніші. Наївний join дав би 112 372 відгуки замість 99 247 (+13.2 %) і змінив би рейтинг
1 264 продавцям.

---

## Підсумок артефактів

| Що | Де |
|---|---|
| Notebook | `HW4.ipynb` (цей файл) |
| Джерело даних, схема, інструкція запуску | `README.md` |
| CSV Olist | `data/olist/*.csv.gz` (6 файлів, 26 МБ) |

**Typed-таблиці Olist:** `olist_customers`, `olist_orders`, `olist_order_items`, `olist_products`,
`olist_sellers`, `olist_order_reviews`, `olist_customer_dim`.
**Службові таблиці:** `seller_alerts`, `seller_score`, `customer_segments`.
**Raw-рівень:** `olist_*_raw` (6 таблиць).

Назви службових таблиць узяті дослівно з умови завдання (`seller_alerts`, `seller_score`,
`customer_segments`), оскільки критерії прийняття посилаються на них саме так.

Notebook не залежить від стану середовища: таблиці перестворюються на початку кожного запуску,
опорна дата для RFM береться з даних, а не з `CURRENT_DATE`, тому *Restart & Run All* відтворює
всі числа з нуля.